In [13]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import recall_score, roc_auc_score
from sklearn.cluster import KMeans
from imblearn.over_sampling import SMOTE
import os
import copy
import warnings
warnings.filterwarnings('ignore')

# ═════════════════════════════════════════════════════════════
# CONFIG — change these per session
# ═════════════════════════════════════════════════════════════
RUN_ONLY_FOLD    = 1       # ← SET THIS: 1, 2, 3, 4, or 5
LABELED_FRACTION = 0.10
ADD_NOISE        = True
NOISE_LEVEL      = 0.3
NOISE_COLS       = ['likesCount', 'videoCount', 'followerCount', 'followingCount']
H_VALUES         = [0.1, 0.2, 0.3, 0.5]

# ── Kaggle dataset names ──────────────────────────────────────
# Run this first in a Kaggle cell to confirm exact names:
#   import os; print(os.listdir('/kaggle/input'))
# Then paste the names below:
KAGGLE_DATASET_LABELLED   = 'Labelled.csv'    # ← CHANGE THIS
KAGGLE_DATASET_UNLABELLED = 'Unlabelled.csv'  # ← CHANGE THIS
# ─────────────────────────────────────────────────────────────

LABELLED_PATH   = '/kaggle/input/datasets/mahsaheidary/labelled-dataset/Labelled.csv'
UNLABELLED_PATH = '/kaggle/input/datasets/mahsaheidary/unlabelled-dataset/Unlabelled.csv'
OUT_PATH        = '/kaggle/working'

OUTPUT_FILE = f'{OUT_PATH}/ssstr_fold_{RUN_ONLY_FOLD}.csv'

print("\n" + "="*60)
print("CONFIG")
print("="*60)
print(f"  Running fold     : {RUN_ONLY_FOLD} of 5")
print(f"  Labeled fraction : {LABELED_FRACTION*100:.0f}%")
print(f"  Add noise        : {ADD_NOISE} (level={NOISE_LEVEL})")
print(f"  Labelled path    : {LABELLED_PATH}")
print(f"  Unlabelled path  : {UNLABELLED_PATH}")
print(f"  Output file      : {OUTPUT_FILE}")
print("="*60)

# ═════════════════════════════════════════════════════════════
# STEP 1: LOAD DATA
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 1: LOADING DATA")
print("="*60)

labeled_full = pd.read_csv(LABELLED_PATH)
unlabeled    = pd.read_csv(UNLABELLED_PATH)

all_features = [
    'likesCount', 'videoCount', 'followerCount', 'followingCount',
    'verified', 'Bio', 'Has_Contact_Info', 'Jaro_Similarity', 'Nickname_Complexity'
]
target = 'Fake'

print(f"Full labeled dataset : {labeled_full.shape}")
print(f"Unlabeled dataset    : {unlabeled.shape}")
print(f"Full class dist:\n{labeled_full[target].value_counts()}")

# ─────────────────────────────────────────────────────────────
# OPTION 1: Reduce labeled pool
# The held-out portion is moved into the unlabeled pool so
# the model can still leverage it via SSSTR.
# ─────────────────────────────────────────────────────────────
if LABELED_FRACTION < 1.0:
    labeled, extra_unlabeled = train_test_split(
        labeled_full,
        train_size=LABELED_FRACTION,
        stratify=labeled_full[target],
        random_state=42
    )
    labeled          = labeled.reset_index(drop=True)
    extra_unlabeled  = extra_unlabeled[all_features].reset_index(drop=True)
    # Append held-out rows (without labels) to unlabeled pool
    unlabeled        = pd.concat(
        [unlabeled[all_features], extra_unlabeled], ignore_index=True
    )
    print(f"\n[Option 1] Using {LABELED_FRACTION*100:.0f}% as labeled → {len(labeled)} rows")
    print(f"  Unlabeled pool now: {len(unlabeled)} rows")
    print(f"  Labeled class dist:\n{labeled[target].value_counts()}")
else:
    labeled = labeled_full.copy()
    print("\n[Option 1] Using 100% of labeled data (no reduction)")

# ─────────────────────────────────────────────────────────────
# OPTION 2: Add Gaussian noise to continuous features
# ─────────────────────────────────────────────────────────────
if ADD_NOISE:
    np.random.seed(42)
    for col in NOISE_COLS:
        std = labeled[col].std()
        labeled[col]   = labeled[col]   + np.random.normal(0, std * NOISE_LEVEL, size=len(labeled))
        unlabeled[col] = unlabeled[col] + np.random.normal(0, std * NOISE_LEVEL, size=len(unlabeled))
    print(f"\n[Option 2] Gaussian noise added to: {NOISE_COLS}")
    print(f"  Noise level: {NOISE_LEVEL} × feature std")
else:
    print("\n[Option 2] No noise added")

# ═════════════════════════════════════════════════════════════
# STEP 2: RFE-SVM FEATURE SELECTION + BIT-FLIP
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 2: FEATURE SELECTION (RFE-SVM + BIT-FLIP)")
print("="*60)

X_fs = labeled[all_features]
y_fs = labeled[target]

scaler_fs   = StandardScaler()
X_fs_scaled = pd.DataFrame(scaler_fs.fit_transform(X_fs), columns=all_features)

# --- RFE-SVM ---
print("\n── Running RFE-SVM ──")
svm = SVC(kernel='linear', random_state=42)
rfe = RFE(estimator=svm, n_features_to_select=1, step=1)
rfe.fit(X_fs_scaled, y_fs)

feature_ranking = pd.DataFrame({
    'Feature': all_features,
    'Ranking': rfe.ranking_
}).sort_values('Ranking')

print("\nFeature Rankings (1 = most important):")
print(feature_ranking.to_string(index=False))

ranked_features = feature_ranking['Feature'].tolist()
subset_3 = ranked_features[:3]
subset_5 = ranked_features[:5]
subset_7 = ranked_features[:7]

print(f"\n3-feature subset: {subset_3}")
print(f"5-feature subset: {subset_5}")
print(f"7-feature subset: {subset_7}")


# --- Bit-Flip Local Search ---
def evaluate_subset(features, X, y):
    """Evaluates feature subset using 5-fold stratified cross-validation."""
    if len(features) == 0:
        return 0.0
    clf = DecisionTreeClassifier(random_state=42)
    cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in cv.split(X[features], y):
        X_tr, X_val = X[features].iloc[train_idx], X[features].iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        sc = StandardScaler()
        X_tr_sc  = sc.fit_transform(X_tr)
        X_val_sc = sc.transform(X_val)
        clf_cv = copy.deepcopy(clf)
        clf_cv.fit(X_tr_sc, y_tr)
        scores.append(clf_cv.score(X_val_sc, y_val))
    return np.mean(scores)


def bit_flip_search(initial_subset, all_features, X, y):
    """Adds or removes one feature at a time to find better subset."""
    print("\n── Running Bit-Flip Local Search ──")
    print(f"  Starting subset  : {initial_subset}")

    current_subset = initial_subset.copy()
    current_score  = evaluate_subset(current_subset, X, y)
    print(f"  Starting accuracy: {current_score:.4f}")

    improved  = True
    iteration = 0

    while improved:
        improved  = False
        iteration += 1
        print(f"\n  [Iteration {iteration}]")
        best_subset = current_subset.copy()
        best_score  = current_score

        print("  Testing removals...")
        for feature in current_subset:
            candidate = [f for f in current_subset if f != feature]
            if len(candidate) == 0:
                continue
            score = evaluate_subset(candidate, X, y)
            print(f"    Remove '{feature}': {score:.4f}")
            if score > best_score:
                best_score  = score
                best_subset = candidate
                improved    = True

        print("  Testing additions...")
        for feature in [f for f in all_features if f not in current_subset]:
            candidate = current_subset + [feature]
            score     = evaluate_subset(candidate, X, y)
            print(f"    Add '{feature}': {score:.4f}")
            if score > best_score:
                best_score  = score
                best_subset = candidate
                improved    = True

        if improved:
            current_subset = best_subset
            current_score  = best_score
            print(f"\n  ✓ Improved → {current_subset} (accuracy={current_score:.4f})")
        else:
            print(f"\n  ✓ No improvement. Stopping.")

    print(f"\n  Best subset  : {current_subset}")
    print(f"  Best accuracy: {current_score:.4f}")
    return current_subset, current_score


best_subset, best_score = bit_flip_search(subset_5, all_features, X_fs, y_fs)

feature_subsets = {
    '3_features'   : subset_3,
    '5_features'   : subset_5,
    '7_features'   : subset_7,
    'best_features': best_subset
}

print("\n" + "="*60)
print("FEATURE SUBSETS SUMMARY")
print("="*60)
for name, feats in feature_subsets.items():
    print(f"  {name:15s}: {feats}")

# ═════════════════════════════════════════════════════════════
# STEP 3: NORMALIZER CLASS
# ═════════════════════════════════════════════════════════════
columns_to_normalize = [
    'likesCount', 'videoCount', 'followerCount', 'followingCount',
    'Jaro_Similarity', 'Nickname_Complexity'
]


class Normalizer:
    def __init__(self):
        self.scaler = None

    def fit_transform(self, df, columns, method='zscore'):
        missing = [c for c in columns if c not in df.columns]
        if missing:
            raise ValueError(f"Columns not found: {missing}")
        if method == 'zscore':
            self.scaler = StandardScaler()
        elif method == 'minmax':
            self.scaler = MinMaxScaler()
        else:
            raise ValueError(f"Unknown method: {method}")
        out          = df.copy()
        out[columns] = self.scaler.fit_transform(df[columns])
        return out

    def transform(self, df, columns):
        if self.scaler is None:
            raise ValueError("Scaler not fitted yet.")
        missing = [c for c in columns if c not in df.columns]
        if missing:
            raise ValueError(f"Columns not found: {missing}")
        out          = df.copy()
        out[columns] = self.scaler.transform(df[columns])
        return out


# ═════════════════════════════════════════════════════════════
# STEP 4: RESAMPLING FUNCTIONS
# ═════════════════════════════════════════════════════════════

def apply_smote(X_train, y_train, random_state=42):
    y_train = pd.Series(y_train).reset_index(drop=True)
    minority_count = y_train.value_counts().min()
    k = min(5, minority_count - 1)
    if k < 1:
        print("  SMOTE skipped: minority class too small")
        return X_train, y_train
    smote        = SMOTE(random_state=random_state, k_neighbors=k)
    X_res, y_res = smote.fit_resample(X_train, y_train)
    print(f"  After SMOTE: {pd.Series(y_res).value_counts().to_dict()}")
    return X_res, pd.Series(y_res).reset_index(drop=True)


def apply_cbute(X_train, y_train, random_state=42):
    X_train = X_train.reset_index(drop=True)
    y_train = pd.Series(y_train).reset_index(drop=True)

    minority_class = y_train.value_counts().idxmin()
    majority_class = y_train.value_counts().idxmax()

    X_min = X_train[y_train == minority_class]
    X_maj = X_train[y_train == majority_class]
    y_min = y_train[y_train == minority_class]

    n_clusters = max(1, len(X_min))
    kmeans     = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    kmeans.fit(X_maj)

    X_maj_reduced = pd.DataFrame(kmeans.cluster_centers_, columns=X_train.columns)
    y_maj_reduced = pd.Series([majority_class] * n_clusters)

    X_res = pd.concat([X_min.reset_index(drop=True), X_maj_reduced], ignore_index=True)
    y_res = pd.Series(
        pd.concat([y_min.reset_index(drop=True), y_maj_reduced], ignore_index=True)
    ).reset_index(drop=True)

    print(f"  After CBUTE: {y_res.value_counts().to_dict()}")
    return X_res, y_res


# ═════════════════════════════════════════════════════════════
# STEP 5: CLASSIFIERS + METRICS
# ═════════════════════════════════════════════════════════════

classifiers = {
    'CART': DecisionTreeClassifier(random_state=42),
    'RF'  : RandomForestClassifier(n_estimators=20, random_state=42),
    'GB'  : GradientBoostingClassifier(n_estimators=20, random_state=42),
    'AB'  : AdaBoostClassifier(random_state=42, algorithm='SAMME'),
    'KNN' : KNeighborsClassifier(),
    'NB'  : GaussianNB()
}


def g_mean(y_true, y_pred):
    recall = recall_score(y_true, y_pred, zero_division=0)
    tnr    = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    return np.sqrt(recall * tnr)


# ═════════════════════════════════════════════════════════════
# STEP 6: SSSTR ALGORITHM
# ═════════════════════════════════════════════════════════════

def run_ssstr(clf, X_train, y_train, X_unlabeled, X_test, y_test, h, verbose=False):
    L_X = X_train.copy().reset_index(drop=True)
    L_y = pd.Series(y_train).copy().reset_index(drop=True)
    U_X = X_unlabeled.copy().reset_index(drop=True)

    iteration = 1
    while len(U_X) > 0:
        model = copy.deepcopy(clf)
        model.fit(L_X, L_y)

        if hasattr(model, 'predict_proba'):
            probs        = model.predict_proba(U_X)
            confidence   = probs.max(axis=1)
            pred_col_idx = probs.argmax(axis=1)
            preds        = np.array([model.classes_[i] for i in pred_col_idx])
        else:
            preds      = model.predict(U_X)
            confidence = np.ones(len(preds))

        n_select = max(1, int(h * len(U_X)))
        top_idx  = np.argsort(confidence)[::-1][:n_select]

        L_X = pd.concat([L_X, U_X.iloc[top_idx]],        ignore_index=True)
        L_y = pd.concat([L_y, pd.Series(preds[top_idx])], ignore_index=True)
        U_X = U_X.drop(index=U_X.index[top_idx]).reset_index(drop=True)

        iteration += 1

    model = copy.deepcopy(clf)
    model.fit(L_X, L_y)

    y_pred = model.predict(X_test)
    y_prob = (model.predict_proba(X_test)[:, 1]
              if hasattr(model, 'predict_proba') else y_pred)

    recall = recall_score(y_test, y_pred, zero_division=0) * 100
    gmean  = g_mean(y_test, y_pred) * 100
    auc    = roc_auc_score(y_test, y_prob) * 100

    return recall, gmean, auc


# ═════════════════════════════════════════════════════════════
# STEP 7: FIND BEST h
# ═════════════════════════════════════════════════════════════

def find_best_h(clf, X_train, y_train, X_unlabeled, X_test, y_test, clf_name):
    print(f"\n  Finding best h for {clf_name}...")
    best_h, best_score, best_metrics = None, -1, None

    for h in H_VALUES:
        recall, gmean, auc = run_ssstr(
            clf, X_train, y_train,
            X_unlabeled, X_test, y_test,
            h, verbose=False
        )
        # Select best h using balanced metrics only (not Recall alone)
        selection_score = (gmean + auc) / 2
        print(f"  h={h} → Recall={recall:.2f}% | "
              f"G-Mean={gmean:.2f}% | AUC={auc:.2f}% | "
              f"Avg(GMean+AUC)={selection_score:.2f}%")

        if selection_score > best_score:
            best_score   = selection_score
            best_h       = h
            best_metrics = (recall, gmean, auc)

    print(f"\n  ✓ Best h={best_h} (Avg={best_score:.2f}%)")
    return best_h, best_metrics


# ═════════════════════════════════════════════════════════════
# STEP 8: RUN EXPERIMENT PER FEATURE SUBSET
# ═════════════════════════════════════════════════════════════

def run_experiment(X_train, y_train,
                   X_unlabeled, X_test, y_test,
                   norm_name, subset_name, features, fold=None):

    fold_str = f" | Fold {fold}" if fold is not None else ""
    print(f"\n{'='*60}")
    print(f"  Normalization : {norm_name}{fold_str}")
    print(f"  Feature Subset: {subset_name} → {features}")
    print(f"{'='*60}")

    results = []

    X_tr_sub  = X_train[features].reset_index(drop=True)
    X_te_sub  = X_test[features].reset_index(drop=True)
    X_unl_sub = X_unlabeled[features].reset_index(drop=True)
    y_tr_nors = pd.Series(y_train).reset_index(drop=True)

    print(f"\n  Applying resampling on {subset_name} ({len(features)} features)...")
    X_tr_smote, y_tr_smote = apply_smote(X_tr_sub, y_tr_nors)
    X_tr_cbute, y_tr_cbute = apply_cbute(X_tr_sub, y_tr_nors)

    sampling_sets = {
        'NORS' : (X_tr_sub,    y_tr_nors),
        'SMOTE': (X_tr_smote,  y_tr_smote),
        'CBUTE': (X_tr_cbute,  y_tr_cbute)
    }

    for clf_name, clf in classifiers.items():
        for sampling_name, (X_tr, y_tr) in sampling_sets.items():
            print(f"\n{'─'*60}")
            print(f"  Classifier: {clf_name} | Resampling: {sampling_name}")
            print(f"  Train size: {len(X_tr)} | "
                  f"Class dist: {pd.Series(y_tr).value_counts().to_dict()}")
            print(f"{'─'*60}")

            best_h, (recall, gmean, auc) = find_best_h(
                clf,
                X_tr, y_tr,
                X_unl_sub,
                X_te_sub, y_test,
                clf_name
            )

            results.append({
                'Fold'         : fold,
                'Normalization': norm_name,
                'Subset'       : subset_name,
                'Classifier'   : clf_name,
                'Resampling'   : sampling_name,
                'Best_h'       : best_h,
                'Recall'       : round(recall, 2),
                'G-Mean'       : round(gmean, 2),
                'AUC'          : round(auc, 2)
            })

            print(f"\n  ✓ {clf_name} | {sampling_name} | {norm_name} | {subset_name}")
            print(f"    Best h={best_h} | Recall={recall:.2f}% | "
                  f"G-Mean={gmean:.2f}% | AUC={auc:.2f}%")

    return pd.DataFrame(results)


# ═════════════════════════════════════════════════════════════
# STEP 9: RUN SINGLE FOLD ONLY
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print(f"STEP 9: RUNNING FOLD {RUN_ONLY_FOLD} OF 5")
print("="*60)

N_FOLDS = 5
skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
X_all   = labeled[all_features]
y_all   = labeled[target]

fold_results = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all), 1):
    if fold != RUN_ONLY_FOLD:
        continue  # skip all other folds

    print(f"\n{'='*60}")
    print(f"  FOLD {fold}/{N_FOLDS}")
    print(f"{'='*60}")

    X_train_raw = X_all.iloc[train_idx].reset_index(drop=True)
    X_test_raw  = X_all.iloc[test_idx].reset_index(drop=True)
    y_train     = y_all.iloc[train_idx].reset_index(drop=True)
    y_test      = y_all.iloc[test_idx].reset_index(drop=True)

    print(f"  Train: {len(X_train_raw)} | Test: {len(X_test_raw)}")
    print(f"  Train dist: {y_train.value_counts().to_dict()}")
    print(f"  Test dist : {y_test.value_counts().to_dict()}")

    # Normalize per fold (fit on train only)
    normZ         = Normalizer()
    X_train_Z     = normZ.fit_transform(X_train_raw, columns_to_normalize, method='zscore')
    X_test_Z      = normZ.transform(X_test_raw,      columns_to_normalize)
    X_unlabeled_Z = normZ.transform(unlabeled[all_features], columns_to_normalize)

    normM         = Normalizer()
    X_train_M     = normM.fit_transform(X_train_raw, columns_to_normalize, method='minmax')
    X_test_M      = normM.transform(X_test_raw,      columns_to_normalize)
    X_unlabeled_M = normM.transform(unlabeled[all_features], columns_to_normalize)

    experiment_configs = {
        'Z-Score': (X_train_Z, X_test_Z, X_unlabeled_Z),
        'Min-Max': (X_train_M, X_test_M, X_unlabeled_M),
    }

    for subset_name, features in feature_subsets.items():
        for norm_name, (X_tr, X_te, X_unl) in experiment_configs.items():
            results = run_experiment(
                X_tr, y_train,
                X_unl, X_te, y_test,
                norm_name, subset_name, features,
                fold=fold
            )
            fold_results.append(results)

            # Checkpoint save after every combination
            pd.concat(fold_results, ignore_index=True).to_csv(
                OUTPUT_FILE, index=False
            )
            print(f"  ✓ Checkpoint saved: Fold {fold} | {subset_name} | {norm_name}")

# ═════════════════════════════════════════════════════════════
# STEP 10: SAVE FOLD RESULTS
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print(f"STEP 10: FOLD {RUN_ONLY_FOLD} RESULTS")
print("="*60)

final = pd.concat(fold_results, ignore_index=True)
final.to_csv(OUTPUT_FILE, index=False)
print(f"\n✓ Results saved to {OUTPUT_FILE}")
print(final.to_string(index=False))

# ═════════════════════════════════════════════════════════════
# MERGE SCRIPT — run this after ALL 5 folds are complete
# ═════════════════════════════════════════════════════════════
# import pandas as pd, glob
#
# files    = sorted(glob.glob(f'{OUT_PATH}/ssstr_fold_*.csv'))
# combined = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
# combined.to_csv(f'{OUT_PATH}/ssstr_cv_all_folds.csv', index=False)
#
# avg_results = combined.groupby(
#     ['Normalization', 'Subset', 'Classifier', 'Resampling']
# )[['Recall', 'G-Mean', 'AUC']].agg(['mean', 'std']).round(2)
#
# avg_results.columns = ['Recall_mean', 'Recall_std',
#                        'GMean_mean',  'GMean_std',
#                        'AUC_mean',    'AUC_std']
# avg_results = avg_results.reset_index()
# avg_results.to_csv(f'{OUT_PATH}/ssstr_cv_results.csv', index=False)
# print(avg_results.to_string(index=False))


CONFIG
  Running fold     : 1 of 5
  Labeled fraction : 10%
  Add noise        : True (level=0.3)
  Labelled path    : /kaggle/input/datasets/mahsaheidary/labelled-dataset/Labelled.csv
  Unlabelled path  : /kaggle/input/datasets/mahsaheidary/unlabelled-dataset/Unlabelled.csv
  Output file      : /kaggle/working/ssstr_fold_1.csv

STEP 1: LOADING DATA
Full labeled dataset : (6873, 11)
Unlabeled dataset    : (23877, 10)
Full class dist:
Fake
1    4020
0    2853
Name: count, dtype: int64

[Option 1] Using 10% as labeled → 687 rows
  Unlabeled pool now: 30063 rows
  Labeled class dist:
Fake
1    402
0    285
Name: count, dtype: int64

[Option 2] Gaussian noise added to: ['likesCount', 'videoCount', 'followerCount', 'followingCount']
  Noise level: 0.3 × feature std

STEP 2: FEATURE SELECTION (RFE-SVM + BIT-FLIP)

── Running RFE-SVM ──

Feature Rankings (1 = most important):
            Feature  Ranking
Nickname_Complexity        1
         videoCount        2
                Bio        3
 

In [2]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import recall_score, roc_auc_score
from sklearn.cluster import KMeans
from imblearn.over_sampling import SMOTE
import os
import copy
import warnings
warnings.filterwarnings('ignore')

# ═════════════════════════════════════════════════════════════
# CONFIG — change these per session
# ═════════════════════════════════════════════════════════════
RUN_ONLY_FOLD    = 2       # ← SET THIS: 1, 2, 3, 4, or 5
LABELED_FRACTION = 0.10
ADD_NOISE        = True
NOISE_LEVEL      = 0.3
NOISE_COLS       = ['likesCount', 'videoCount', 'followerCount', 'followingCount']
H_VALUES         = [0.1, 0.2, 0.3, 0.5]

# ── Kaggle dataset names ──────────────────────────────────────
# Run this first in a Kaggle cell to confirm exact names:
#   import os; print(os.listdir('/kaggle/input'))
# Then paste the names below:
KAGGLE_DATASET_LABELLED   = 'Labelled.csv'    # ← CHANGE THIS
KAGGLE_DATASET_UNLABELLED = 'Unlabelled.csv'  # ← CHANGE THIS
# ─────────────────────────────────────────────────────────────

LABELLED_PATH   = '/kaggle/input/datasets/mahsaheidary/labelled-dataset/Labelled.csv'
UNLABELLED_PATH = '/kaggle/input/datasets/mahsaheidary/unlabelled-dataset/Unlabelled.csv'
OUT_PATH        = '/kaggle/working'

OUTPUT_FILE = f'{OUT_PATH}/ssstr_fold_{RUN_ONLY_FOLD}.csv'

print("\n" + "="*60)
print("CONFIG")
print("="*60)
print(f"  Running fold     : {RUN_ONLY_FOLD} of 5")
print(f"  Labeled fraction : {LABELED_FRACTION*100:.0f}%")
print(f"  Add noise        : {ADD_NOISE} (level={NOISE_LEVEL})")
print(f"  Labelled path    : {LABELLED_PATH}")
print(f"  Unlabelled path  : {UNLABELLED_PATH}")
print(f"  Output file      : {OUTPUT_FILE}")
print("="*60)

# ═════════════════════════════════════════════════════════════
# STEP 1: LOAD DATA
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 1: LOADING DATA")
print("="*60)

labeled_full = pd.read_csv(LABELLED_PATH)
unlabeled    = pd.read_csv(UNLABELLED_PATH)

all_features = [
    'likesCount', 'videoCount', 'followerCount', 'followingCount',
    'verified', 'Bio', 'Has_Contact_Info', 'Jaro_Similarity', 'Nickname_Complexity'
]
target = 'Fake'

print(f"Full labeled dataset : {labeled_full.shape}")
print(f"Unlabeled dataset    : {unlabeled.shape}")
print(f"Full class dist:\n{labeled_full[target].value_counts()}")

# ─────────────────────────────────────────────────────────────
# OPTION 1: Reduce labeled pool
# The held-out portion is moved into the unlabeled pool so
# the model can still leverage it via SSSTR.
# ─────────────────────────────────────────────────────────────
if LABELED_FRACTION < 1.0:
    labeled, extra_unlabeled = train_test_split(
        labeled_full,
        train_size=LABELED_FRACTION,
        stratify=labeled_full[target],
        random_state=42
    )
    labeled          = labeled.reset_index(drop=True)
    extra_unlabeled  = extra_unlabeled[all_features].reset_index(drop=True)
    # Append held-out rows (without labels) to unlabeled pool
    unlabeled        = pd.concat(
        [unlabeled[all_features], extra_unlabeled], ignore_index=True
    )
    print(f"\n[Option 1] Using {LABELED_FRACTION*100:.0f}% as labeled → {len(labeled)} rows")
    print(f"  Unlabeled pool now: {len(unlabeled)} rows")
    print(f"  Labeled class dist:\n{labeled[target].value_counts()}")
else:
    labeled = labeled_full.copy()
    print("\n[Option 1] Using 100% of labeled data (no reduction)")

# ─────────────────────────────────────────────────────────────
# OPTION 2: Add Gaussian noise to continuous features
# ─────────────────────────────────────────────────────────────
if ADD_NOISE:
    np.random.seed(42)
    for col in NOISE_COLS:
        std = labeled[col].std()
        labeled[col]   = labeled[col]   + np.random.normal(0, std * NOISE_LEVEL, size=len(labeled))
        unlabeled[col] = unlabeled[col] + np.random.normal(0, std * NOISE_LEVEL, size=len(unlabeled))
    print(f"\n[Option 2] Gaussian noise added to: {NOISE_COLS}")
    print(f"  Noise level: {NOISE_LEVEL} × feature std")
else:
    print("\n[Option 2] No noise added")

# ═════════════════════════════════════════════════════════════
# STEP 2: RFE-SVM FEATURE SELECTION + BIT-FLIP
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 2: FEATURE SELECTION (RFE-SVM + BIT-FLIP)")
print("="*60)

X_fs = labeled[all_features]
y_fs = labeled[target]

scaler_fs   = StandardScaler()
X_fs_scaled = pd.DataFrame(scaler_fs.fit_transform(X_fs), columns=all_features)

# --- RFE-SVM ---
print("\n── Running RFE-SVM ──")
svm = SVC(kernel='linear', random_state=42)
rfe = RFE(estimator=svm, n_features_to_select=1, step=1)
rfe.fit(X_fs_scaled, y_fs)

feature_ranking = pd.DataFrame({
    'Feature': all_features,
    'Ranking': rfe.ranking_
}).sort_values('Ranking')

print("\nFeature Rankings (1 = most important):")
print(feature_ranking.to_string(index=False))

ranked_features = feature_ranking['Feature'].tolist()
subset_3 = ranked_features[:3]
subset_5 = ranked_features[:5]
subset_7 = ranked_features[:7]

print(f"\n3-feature subset: {subset_3}")
print(f"5-feature subset: {subset_5}")
print(f"7-feature subset: {subset_7}")


# --- Bit-Flip Local Search ---
def evaluate_subset(features, X, y):
    """Evaluates feature subset using 5-fold stratified cross-validation."""
    if len(features) == 0:
        return 0.0
    clf = DecisionTreeClassifier(random_state=42)
    cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in cv.split(X[features], y):
        X_tr, X_val = X[features].iloc[train_idx], X[features].iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        sc = StandardScaler()
        X_tr_sc  = sc.fit_transform(X_tr)
        X_val_sc = sc.transform(X_val)
        clf_cv = copy.deepcopy(clf)
        clf_cv.fit(X_tr_sc, y_tr)
        scores.append(clf_cv.score(X_val_sc, y_val))
    return np.mean(scores)


def bit_flip_search(initial_subset, all_features, X, y):
    """Adds or removes one feature at a time to find better subset."""
    print("\n── Running Bit-Flip Local Search ──")
    print(f"  Starting subset  : {initial_subset}")

    current_subset = initial_subset.copy()
    current_score  = evaluate_subset(current_subset, X, y)
    print(f"  Starting accuracy: {current_score:.4f}")

    improved  = True
    iteration = 0

    while improved:
        improved  = False
        iteration += 1
        print(f"\n  [Iteration {iteration}]")
        best_subset = current_subset.copy()
        best_score  = current_score

        print("  Testing removals...")
        for feature in current_subset:
            candidate = [f for f in current_subset if f != feature]
            if len(candidate) == 0:
                continue
            score = evaluate_subset(candidate, X, y)
            print(f"    Remove '{feature}': {score:.4f}")
            if score > best_score:
                best_score  = score
                best_subset = candidate
                improved    = True

        print("  Testing additions...")
        for feature in [f for f in all_features if f not in current_subset]:
            candidate = current_subset + [feature]
            score     = evaluate_subset(candidate, X, y)
            print(f"    Add '{feature}': {score:.4f}")
            if score > best_score:
                best_score  = score
                best_subset = candidate
                improved    = True

        if improved:
            current_subset = best_subset
            current_score  = best_score
            print(f"\n  ✓ Improved → {current_subset} (accuracy={current_score:.4f})")
        else:
            print(f"\n  ✓ No improvement. Stopping.")

    print(f"\n  Best subset  : {current_subset}")
    print(f"  Best accuracy: {current_score:.4f}")
    return current_subset, current_score


best_subset, best_score = bit_flip_search(subset_5, all_features, X_fs, y_fs)

feature_subsets = {
    '3_features'   : subset_3,
    '5_features'   : subset_5,
    '7_features'   : subset_7,
    'best_features': best_subset
}

print("\n" + "="*60)
print("FEATURE SUBSETS SUMMARY")
print("="*60)
for name, feats in feature_subsets.items():
    print(f"  {name:15s}: {feats}")

# ═════════════════════════════════════════════════════════════
# STEP 3: NORMALIZER CLASS
# ═════════════════════════════════════════════════════════════
columns_to_normalize = [
    'likesCount', 'videoCount', 'followerCount', 'followingCount',
    'Jaro_Similarity', 'Nickname_Complexity'
]


class Normalizer:
    def __init__(self):
        self.scaler = None

    def fit_transform(self, df, columns, method='zscore'):
        missing = [c for c in columns if c not in df.columns]
        if missing:
            raise ValueError(f"Columns not found: {missing}")
        if method == 'zscore':
            self.scaler = StandardScaler()
        elif method == 'minmax':
            self.scaler = MinMaxScaler()
        else:
            raise ValueError(f"Unknown method: {method}")
        out          = df.copy()
        out[columns] = self.scaler.fit_transform(df[columns])
        return out

    def transform(self, df, columns):
        if self.scaler is None:
            raise ValueError("Scaler not fitted yet.")
        missing = [c for c in columns if c not in df.columns]
        if missing:
            raise ValueError(f"Columns not found: {missing}")
        out          = df.copy()
        out[columns] = self.scaler.transform(df[columns])
        return out


# ═════════════════════════════════════════════════════════════
# STEP 4: RESAMPLING FUNCTIONS
# ═════════════════════════════════════════════════════════════

def apply_smote(X_train, y_train, random_state=42):
    y_train = pd.Series(y_train).reset_index(drop=True)
    minority_count = y_train.value_counts().min()
    k = min(5, minority_count - 1)
    if k < 1:
        print("  SMOTE skipped: minority class too small")
        return X_train, y_train
    smote        = SMOTE(random_state=random_state, k_neighbors=k)
    X_res, y_res = smote.fit_resample(X_train, y_train)
    print(f"  After SMOTE: {pd.Series(y_res).value_counts().to_dict()}")
    return X_res, pd.Series(y_res).reset_index(drop=True)


def apply_cbute(X_train, y_train, random_state=42):
    X_train = X_train.reset_index(drop=True)
    y_train = pd.Series(y_train).reset_index(drop=True)

    minority_class = y_train.value_counts().idxmin()
    majority_class = y_train.value_counts().idxmax()

    X_min = X_train[y_train == minority_class]
    X_maj = X_train[y_train == majority_class]
    y_min = y_train[y_train == minority_class]

    n_clusters = max(1, len(X_min))
    kmeans     = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    kmeans.fit(X_maj)

    X_maj_reduced = pd.DataFrame(kmeans.cluster_centers_, columns=X_train.columns)
    y_maj_reduced = pd.Series([majority_class] * n_clusters)

    X_res = pd.concat([X_min.reset_index(drop=True), X_maj_reduced], ignore_index=True)
    y_res = pd.Series(
        pd.concat([y_min.reset_index(drop=True), y_maj_reduced], ignore_index=True)
    ).reset_index(drop=True)

    print(f"  After CBUTE: {y_res.value_counts().to_dict()}")
    return X_res, y_res


# ═════════════════════════════════════════════════════════════
# STEP 5: CLASSIFIERS + METRICS
# ═════════════════════════════════════════════════════════════

classifiers = {
    'CART': DecisionTreeClassifier(random_state=42),
    'RF'  : RandomForestClassifier(n_estimators=20, random_state=42),
    'GB'  : GradientBoostingClassifier(n_estimators=20, random_state=42),
    'AB'  : AdaBoostClassifier(random_state=42, algorithm='SAMME'),
    'KNN' : KNeighborsClassifier(),
    'NB'  : GaussianNB()
}


def g_mean(y_true, y_pred):
    recall = recall_score(y_true, y_pred, zero_division=0)
    tnr    = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    return np.sqrt(recall * tnr)


# ═════════════════════════════════════════════════════════════
# STEP 6: SSSTR ALGORITHM
# ═════════════════════════════════════════════════════════════

def run_ssstr(clf, X_train, y_train, X_unlabeled, X_test, y_test, h, verbose=False):
    L_X = X_train.copy().reset_index(drop=True)
    L_y = pd.Series(y_train).copy().reset_index(drop=True)
    U_X = X_unlabeled.copy().reset_index(drop=True)

    iteration = 1
    while len(U_X) > 0:
        model = copy.deepcopy(clf)
        model.fit(L_X, L_y)

        if hasattr(model, 'predict_proba'):
            probs        = model.predict_proba(U_X)
            confidence   = probs.max(axis=1)
            pred_col_idx = probs.argmax(axis=1)
            preds        = np.array([model.classes_[i] for i in pred_col_idx])
        else:
            preds      = model.predict(U_X)
            confidence = np.ones(len(preds))

        n_select = max(1, int(h * len(U_X)))
        top_idx  = np.argsort(confidence)[::-1][:n_select]

        L_X = pd.concat([L_X, U_X.iloc[top_idx]],        ignore_index=True)
        L_y = pd.concat([L_y, pd.Series(preds[top_idx])], ignore_index=True)
        U_X = U_X.drop(index=U_X.index[top_idx]).reset_index(drop=True)

        iteration += 1

    model = copy.deepcopy(clf)
    model.fit(L_X, L_y)

    y_pred = model.predict(X_test)
    y_prob = (model.predict_proba(X_test)[:, 1]
              if hasattr(model, 'predict_proba') else y_pred)

    recall = recall_score(y_test, y_pred, zero_division=0) * 100
    gmean  = g_mean(y_test, y_pred) * 100
    auc    = roc_auc_score(y_test, y_prob) * 100

    return recall, gmean, auc


# ═════════════════════════════════════════════════════════════
# STEP 7: FIND BEST h
# ═════════════════════════════════════════════════════════════

def find_best_h(clf, X_train, y_train, X_unlabeled, X_test, y_test, clf_name):
    print(f"\n  Finding best h for {clf_name}...")
    best_h, best_score, best_metrics = None, -1, None

    for h in H_VALUES:
        recall, gmean, auc = run_ssstr(
            clf, X_train, y_train,
            X_unlabeled, X_test, y_test,
            h, verbose=False
        )
        # Select best h using balanced metrics only (not Recall alone)
        selection_score = (gmean + auc) / 2
        print(f"  h={h} → Recall={recall:.2f}% | "
              f"G-Mean={gmean:.2f}% | AUC={auc:.2f}% | "
              f"Avg(GMean+AUC)={selection_score:.2f}%")

        if selection_score > best_score:
            best_score   = selection_score
            best_h       = h
            best_metrics = (recall, gmean, auc)

    print(f"\n  ✓ Best h={best_h} (Avg={best_score:.2f}%)")
    return best_h, best_metrics


# ═════════════════════════════════════════════════════════════
# STEP 8: RUN EXPERIMENT PER FEATURE SUBSET
# ═════════════════════════════════════════════════════════════

def run_experiment(X_train, y_train,
                   X_unlabeled, X_test, y_test,
                   norm_name, subset_name, features, fold=None):

    fold_str = f" | Fold {fold}" if fold is not None else ""
    print(f"\n{'='*60}")
    print(f"  Normalization : {norm_name}{fold_str}")
    print(f"  Feature Subset: {subset_name} → {features}")
    print(f"{'='*60}")

    results = []

    X_tr_sub  = X_train[features].reset_index(drop=True)
    X_te_sub  = X_test[features].reset_index(drop=True)
    X_unl_sub = X_unlabeled[features].reset_index(drop=True)
    y_tr_nors = pd.Series(y_train).reset_index(drop=True)

    print(f"\n  Applying resampling on {subset_name} ({len(features)} features)...")
    X_tr_smote, y_tr_smote = apply_smote(X_tr_sub, y_tr_nors)
    X_tr_cbute, y_tr_cbute = apply_cbute(X_tr_sub, y_tr_nors)

    sampling_sets = {
        'NORS' : (X_tr_sub,    y_tr_nors),
        'SMOTE': (X_tr_smote,  y_tr_smote),
        'CBUTE': (X_tr_cbute,  y_tr_cbute)
    }

    for clf_name, clf in classifiers.items():
        for sampling_name, (X_tr, y_tr) in sampling_sets.items():
            print(f"\n{'─'*60}")
            print(f"  Classifier: {clf_name} | Resampling: {sampling_name}")
            print(f"  Train size: {len(X_tr)} | "
                  f"Class dist: {pd.Series(y_tr).value_counts().to_dict()}")
            print(f"{'─'*60}")

            best_h, (recall, gmean, auc) = find_best_h(
                clf,
                X_tr, y_tr,
                X_unl_sub,
                X_te_sub, y_test,
                clf_name
            )

            results.append({
                'Fold'         : fold,
                'Normalization': norm_name,
                'Subset'       : subset_name,
                'Classifier'   : clf_name,
                'Resampling'   : sampling_name,
                'Best_h'       : best_h,
                'Recall'       : round(recall, 2),
                'G-Mean'       : round(gmean, 2),
                'AUC'          : round(auc, 2)
            })

            print(f"\n  ✓ {clf_name} | {sampling_name} | {norm_name} | {subset_name}")
            print(f"    Best h={best_h} | Recall={recall:.2f}% | "
                  f"G-Mean={gmean:.2f}% | AUC={auc:.2f}%")

    return pd.DataFrame(results)


# ═════════════════════════════════════════════════════════════
# STEP 9: RUN SINGLE FOLD ONLY
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print(f"STEP 9: RUNNING FOLD {RUN_ONLY_FOLD} OF 5")
print("="*60)

N_FOLDS = 5
skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
X_all   = labeled[all_features]
y_all   = labeled[target]

fold_results = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all), 1):
    if fold != RUN_ONLY_FOLD:
        continue  # skip all other folds

    print(f"\n{'='*60}")
    print(f"  FOLD {fold}/{N_FOLDS}")
    print(f"{'='*60}")

    X_train_raw = X_all.iloc[train_idx].reset_index(drop=True)
    X_test_raw  = X_all.iloc[test_idx].reset_index(drop=True)
    y_train     = y_all.iloc[train_idx].reset_index(drop=True)
    y_test      = y_all.iloc[test_idx].reset_index(drop=True)

    print(f"  Train: {len(X_train_raw)} | Test: {len(X_test_raw)}")
    print(f"  Train dist: {y_train.value_counts().to_dict()}")
    print(f"  Test dist : {y_test.value_counts().to_dict()}")

    # Normalize per fold (fit on train only)
    normZ         = Normalizer()
    X_train_Z     = normZ.fit_transform(X_train_raw, columns_to_normalize, method='zscore')
    X_test_Z      = normZ.transform(X_test_raw,      columns_to_normalize)
    X_unlabeled_Z = normZ.transform(unlabeled[all_features], columns_to_normalize)

    normM         = Normalizer()
    X_train_M     = normM.fit_transform(X_train_raw, columns_to_normalize, method='minmax')
    X_test_M      = normM.transform(X_test_raw,      columns_to_normalize)
    X_unlabeled_M = normM.transform(unlabeled[all_features], columns_to_normalize)

    experiment_configs = {
        'Z-Score': (X_train_Z, X_test_Z, X_unlabeled_Z),
        'Min-Max': (X_train_M, X_test_M, X_unlabeled_M),
    }

    for subset_name, features in feature_subsets.items():
        for norm_name, (X_tr, X_te, X_unl) in experiment_configs.items():
            results = run_experiment(
                X_tr, y_train,
                X_unl, X_te, y_test,
                norm_name, subset_name, features,
                fold=fold
            )
            fold_results.append(results)

            # Checkpoint save after every combination
            pd.concat(fold_results, ignore_index=True).to_csv(
                OUTPUT_FILE, index=False
            )
            print(f"  ✓ Checkpoint saved: Fold {fold} | {subset_name} | {norm_name}")

# ═════════════════════════════════════════════════════════════
# STEP 10: SAVE FOLD RESULTS
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print(f"STEP 10: FOLD {RUN_ONLY_FOLD} RESULTS")
print("="*60)

final = pd.concat(fold_results, ignore_index=True)
final.to_csv(OUTPUT_FILE, index=False)
print(f"\n✓ Results saved to {OUTPUT_FILE}")
print(final.to_string(index=False))

# ═════════════════════════════════════════════════════════════
# MERGE SCRIPT — run this after ALL 5 folds are complete
# ═════════════════════════════════════════════════════════════
# import pandas as pd, glob
#
# files    = sorted(glob.glob(f'{OUT_PATH}/ssstr_fold_*.csv'))
# combined = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
# combined.to_csv(f'{OUT_PATH}/ssstr_cv_all_folds.csv', index=False)
#
# avg_results = combined.groupby(
#     ['Normalization', 'Subset', 'Classifier', 'Resampling']
# )[['Recall', 'G-Mean', 'AUC']].agg(['mean', 'std']).round(2)
#
# avg_results.columns = ['Recall_mean', 'Recall_std',
#                        'GMean_mean',  'GMean_std',
#                        'AUC_mean',    'AUC_std']
# avg_results = avg_results.reset_index()
# avg_results.to_csv(f'{OUT_PATH}/ssstr_cv_results.csv', index=False)
# print(avg_results.to_string(index=False))


CONFIG
  Running fold     : 2 of 5
  Labeled fraction : 10%
  Add noise        : True (level=0.3)
  Labelled path    : /kaggle/input/datasets/mahsaheidary/labelled-dataset/Labelled.csv
  Unlabelled path  : /kaggle/input/datasets/mahsaheidary/unlabelled-dataset/Unlabelled.csv
  Output file      : /kaggle/working/ssstr_fold_2.csv

STEP 1: LOADING DATA
Full labeled dataset : (6873, 11)
Unlabeled dataset    : (23877, 10)
Full class dist:
Fake
1    4020
0    2853
Name: count, dtype: int64

[Option 1] Using 10% as labeled → 687 rows
  Unlabeled pool now: 30063 rows
  Labeled class dist:
Fake
1    402
0    285
Name: count, dtype: int64

[Option 2] Gaussian noise added to: ['likesCount', 'videoCount', 'followerCount', 'followingCount']
  Noise level: 0.3 × feature std

STEP 2: FEATURE SELECTION (RFE-SVM + BIT-FLIP)

── Running RFE-SVM ──

Feature Rankings (1 = most important):
            Feature  Ranking
Nickname_Complexity        1
         videoCount        2
                Bio        3
 

In [3]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import recall_score, roc_auc_score
from sklearn.cluster import KMeans
from imblearn.over_sampling import SMOTE
import os
import copy
import warnings
warnings.filterwarnings('ignore')

# ═════════════════════════════════════════════════════════════
# CONFIG — change these per session
# ═════════════════════════════════════════════════════════════
RUN_ONLY_FOLD    = 3       # ← SET THIS: 1, 2, 3, 4, or 5
LABELED_FRACTION = 0.10
ADD_NOISE        = True
NOISE_LEVEL      = 0.3
NOISE_COLS       = ['likesCount', 'videoCount', 'followerCount', 'followingCount']
H_VALUES         = [0.1, 0.2, 0.3, 0.5]

# ── Kaggle dataset names ──────────────────────────────────────
# Run this first in a Kaggle cell to confirm exact names:
#   import os; print(os.listdir('/kaggle/input'))
# Then paste the names below:
KAGGLE_DATASET_LABELLED   = 'Labelled.csv'    # ← CHANGE THIS
KAGGLE_DATASET_UNLABELLED = 'Unlabelled.csv'  # ← CHANGE THIS
# ─────────────────────────────────────────────────────────────

LABELLED_PATH   = '/kaggle/input/datasets/mahsaheidary/labelled-dataset/Labelled.csv'
UNLABELLED_PATH = '/kaggle/input/datasets/mahsaheidary/unlabelled-dataset/Unlabelled.csv'
OUT_PATH        = '/kaggle/working'

OUTPUT_FILE = f'{OUT_PATH}/ssstr_fold_{RUN_ONLY_FOLD}.csv'

print("\n" + "="*60)
print("CONFIG")
print("="*60)
print(f"  Running fold     : {RUN_ONLY_FOLD} of 5")
print(f"  Labeled fraction : {LABELED_FRACTION*100:.0f}%")
print(f"  Add noise        : {ADD_NOISE} (level={NOISE_LEVEL})")
print(f"  Labelled path    : {LABELLED_PATH}")
print(f"  Unlabelled path  : {UNLABELLED_PATH}")
print(f"  Output file      : {OUTPUT_FILE}")
print("="*60)

# ═════════════════════════════════════════════════════════════
# STEP 1: LOAD DATA
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 1: LOADING DATA")
print("="*60)

labeled_full = pd.read_csv(LABELLED_PATH)
unlabeled    = pd.read_csv(UNLABELLED_PATH)

all_features = [
    'likesCount', 'videoCount', 'followerCount', 'followingCount',
    'verified', 'Bio', 'Has_Contact_Info', 'Jaro_Similarity', 'Nickname_Complexity'
]
target = 'Fake'

print(f"Full labeled dataset : {labeled_full.shape}")
print(f"Unlabeled dataset    : {unlabeled.shape}")
print(f"Full class dist:\n{labeled_full[target].value_counts()}")

# ─────────────────────────────────────────────────────────────
# OPTION 1: Reduce labeled pool
# The held-out portion is moved into the unlabeled pool so
# the model can still leverage it via SSSTR.
# ─────────────────────────────────────────────────────────────
if LABELED_FRACTION < 1.0:
    labeled, extra_unlabeled = train_test_split(
        labeled_full,
        train_size=LABELED_FRACTION,
        stratify=labeled_full[target],
        random_state=42
    )
    labeled          = labeled.reset_index(drop=True)
    extra_unlabeled  = extra_unlabeled[all_features].reset_index(drop=True)
    # Append held-out rows (without labels) to unlabeled pool
    unlabeled        = pd.concat(
        [unlabeled[all_features], extra_unlabeled], ignore_index=True
    )
    print(f"\n[Option 1] Using {LABELED_FRACTION*100:.0f}% as labeled → {len(labeled)} rows")
    print(f"  Unlabeled pool now: {len(unlabeled)} rows")
    print(f"  Labeled class dist:\n{labeled[target].value_counts()}")
else:
    labeled = labeled_full.copy()
    print("\n[Option 1] Using 100% of labeled data (no reduction)")

# ─────────────────────────────────────────────────────────────
# OPTION 2: Add Gaussian noise to continuous features
# ─────────────────────────────────────────────────────────────
if ADD_NOISE:
    np.random.seed(42)
    for col in NOISE_COLS:
        std = labeled[col].std()
        labeled[col]   = labeled[col]   + np.random.normal(0, std * NOISE_LEVEL, size=len(labeled))
        unlabeled[col] = unlabeled[col] + np.random.normal(0, std * NOISE_LEVEL, size=len(unlabeled))
    print(f"\n[Option 2] Gaussian noise added to: {NOISE_COLS}")
    print(f"  Noise level: {NOISE_LEVEL} × feature std")
else:
    print("\n[Option 2] No noise added")

# ═════════════════════════════════════════════════════════════
# STEP 2: RFE-SVM FEATURE SELECTION + BIT-FLIP
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 2: FEATURE SELECTION (RFE-SVM + BIT-FLIP)")
print("="*60)

X_fs = labeled[all_features]
y_fs = labeled[target]

scaler_fs   = StandardScaler()
X_fs_scaled = pd.DataFrame(scaler_fs.fit_transform(X_fs), columns=all_features)

# --- RFE-SVM ---
print("\n── Running RFE-SVM ──")
svm = SVC(kernel='linear', random_state=42)
rfe = RFE(estimator=svm, n_features_to_select=1, step=1)
rfe.fit(X_fs_scaled, y_fs)

feature_ranking = pd.DataFrame({
    'Feature': all_features,
    'Ranking': rfe.ranking_
}).sort_values('Ranking')

print("\nFeature Rankings (1 = most important):")
print(feature_ranking.to_string(index=False))

ranked_features = feature_ranking['Feature'].tolist()
subset_3 = ranked_features[:3]
subset_5 = ranked_features[:5]
subset_7 = ranked_features[:7]

print(f"\n3-feature subset: {subset_3}")
print(f"5-feature subset: {subset_5}")
print(f"7-feature subset: {subset_7}")


# --- Bit-Flip Local Search ---
def evaluate_subset(features, X, y):
    """Evaluates feature subset using 5-fold stratified cross-validation."""
    if len(features) == 0:
        return 0.0
    clf = DecisionTreeClassifier(random_state=42)
    cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in cv.split(X[features], y):
        X_tr, X_val = X[features].iloc[train_idx], X[features].iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        sc = StandardScaler()
        X_tr_sc  = sc.fit_transform(X_tr)
        X_val_sc = sc.transform(X_val)
        clf_cv = copy.deepcopy(clf)
        clf_cv.fit(X_tr_sc, y_tr)
        scores.append(clf_cv.score(X_val_sc, y_val))
    return np.mean(scores)


def bit_flip_search(initial_subset, all_features, X, y):
    """Adds or removes one feature at a time to find better subset."""
    print("\n── Running Bit-Flip Local Search ──")
    print(f"  Starting subset  : {initial_subset}")

    current_subset = initial_subset.copy()
    current_score  = evaluate_subset(current_subset, X, y)
    print(f"  Starting accuracy: {current_score:.4f}")

    improved  = True
    iteration = 0

    while improved:
        improved  = False
        iteration += 1
        print(f"\n  [Iteration {iteration}]")
        best_subset = current_subset.copy()
        best_score  = current_score

        print("  Testing removals...")
        for feature in current_subset:
            candidate = [f for f in current_subset if f != feature]
            if len(candidate) == 0:
                continue
            score = evaluate_subset(candidate, X, y)
            print(f"    Remove '{feature}': {score:.4f}")
            if score > best_score:
                best_score  = score
                best_subset = candidate
                improved    = True

        print("  Testing additions...")
        for feature in [f for f in all_features if f not in current_subset]:
            candidate = current_subset + [feature]
            score     = evaluate_subset(candidate, X, y)
            print(f"    Add '{feature}': {score:.4f}")
            if score > best_score:
                best_score  = score
                best_subset = candidate
                improved    = True

        if improved:
            current_subset = best_subset
            current_score  = best_score
            print(f"\n  ✓ Improved → {current_subset} (accuracy={current_score:.4f})")
        else:
            print(f"\n  ✓ No improvement. Stopping.")

    print(f"\n  Best subset  : {current_subset}")
    print(f"  Best accuracy: {current_score:.4f}")
    return current_subset, current_score


best_subset, best_score = bit_flip_search(subset_5, all_features, X_fs, y_fs)

feature_subsets = {
    '3_features'   : subset_3,
    '5_features'   : subset_5,
    '7_features'   : subset_7,
    'best_features': best_subset
}

print("\n" + "="*60)
print("FEATURE SUBSETS SUMMARY")
print("="*60)
for name, feats in feature_subsets.items():
    print(f"  {name:15s}: {feats}")

# ═════════════════════════════════════════════════════════════
# STEP 3: NORMALIZER CLASS
# ═════════════════════════════════════════════════════════════
columns_to_normalize = [
    'likesCount', 'videoCount', 'followerCount', 'followingCount',
    'Jaro_Similarity', 'Nickname_Complexity'
]


class Normalizer:
    def __init__(self):
        self.scaler = None

    def fit_transform(self, df, columns, method='zscore'):
        missing = [c for c in columns if c not in df.columns]
        if missing:
            raise ValueError(f"Columns not found: {missing}")
        if method == 'zscore':
            self.scaler = StandardScaler()
        elif method == 'minmax':
            self.scaler = MinMaxScaler()
        else:
            raise ValueError(f"Unknown method: {method}")
        out          = df.copy()
        out[columns] = self.scaler.fit_transform(df[columns])
        return out

    def transform(self, df, columns):
        if self.scaler is None:
            raise ValueError("Scaler not fitted yet.")
        missing = [c for c in columns if c not in df.columns]
        if missing:
            raise ValueError(f"Columns not found: {missing}")
        out          = df.copy()
        out[columns] = self.scaler.transform(df[columns])
        return out


# ═════════════════════════════════════════════════════════════
# STEP 4: RESAMPLING FUNCTIONS
# ═════════════════════════════════════════════════════════════

def apply_smote(X_train, y_train, random_state=42):
    y_train = pd.Series(y_train).reset_index(drop=True)
    minority_count = y_train.value_counts().min()
    k = min(5, minority_count - 1)
    if k < 1:
        print("  SMOTE skipped: minority class too small")
        return X_train, y_train
    smote        = SMOTE(random_state=random_state, k_neighbors=k)
    X_res, y_res = smote.fit_resample(X_train, y_train)
    print(f"  After SMOTE: {pd.Series(y_res).value_counts().to_dict()}")
    return X_res, pd.Series(y_res).reset_index(drop=True)


def apply_cbute(X_train, y_train, random_state=42):
    X_train = X_train.reset_index(drop=True)
    y_train = pd.Series(y_train).reset_index(drop=True)

    minority_class = y_train.value_counts().idxmin()
    majority_class = y_train.value_counts().idxmax()

    X_min = X_train[y_train == minority_class]
    X_maj = X_train[y_train == majority_class]
    y_min = y_train[y_train == minority_class]

    n_clusters = max(1, len(X_min))
    kmeans     = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    kmeans.fit(X_maj)

    X_maj_reduced = pd.DataFrame(kmeans.cluster_centers_, columns=X_train.columns)
    y_maj_reduced = pd.Series([majority_class] * n_clusters)

    X_res = pd.concat([X_min.reset_index(drop=True), X_maj_reduced], ignore_index=True)
    y_res = pd.Series(
        pd.concat([y_min.reset_index(drop=True), y_maj_reduced], ignore_index=True)
    ).reset_index(drop=True)

    print(f"  After CBUTE: {y_res.value_counts().to_dict()}")
    return X_res, y_res


# ═════════════════════════════════════════════════════════════
# STEP 5: CLASSIFIERS + METRICS
# ═════════════════════════════════════════════════════════════

classifiers = {
    'CART': DecisionTreeClassifier(random_state=42),
    'RF'  : RandomForestClassifier(n_estimators=20, random_state=42),
    'GB'  : GradientBoostingClassifier(n_estimators=20, random_state=42),
    'AB'  : AdaBoostClassifier(random_state=42, algorithm='SAMME'),
    'KNN' : KNeighborsClassifier(),
    'NB'  : GaussianNB()
}


def g_mean(y_true, y_pred):
    recall = recall_score(y_true, y_pred, zero_division=0)
    tnr    = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    return np.sqrt(recall * tnr)


# ═════════════════════════════════════════════════════════════
# STEP 6: SSSTR ALGORITHM
# ═════════════════════════════════════════════════════════════

def run_ssstr(clf, X_train, y_train, X_unlabeled, X_test, y_test, h, verbose=False):
    L_X = X_train.copy().reset_index(drop=True)
    L_y = pd.Series(y_train).copy().reset_index(drop=True)
    U_X = X_unlabeled.copy().reset_index(drop=True)

    iteration = 1
    while len(U_X) > 0:
        model = copy.deepcopy(clf)
        model.fit(L_X, L_y)

        if hasattr(model, 'predict_proba'):
            probs        = model.predict_proba(U_X)
            confidence   = probs.max(axis=1)
            pred_col_idx = probs.argmax(axis=1)
            preds        = np.array([model.classes_[i] for i in pred_col_idx])
        else:
            preds      = model.predict(U_X)
            confidence = np.ones(len(preds))

        n_select = max(1, int(h * len(U_X)))
        top_idx  = np.argsort(confidence)[::-1][:n_select]

        L_X = pd.concat([L_X, U_X.iloc[top_idx]],        ignore_index=True)
        L_y = pd.concat([L_y, pd.Series(preds[top_idx])], ignore_index=True)
        U_X = U_X.drop(index=U_X.index[top_idx]).reset_index(drop=True)

        iteration += 1

    model = copy.deepcopy(clf)
    model.fit(L_X, L_y)

    y_pred = model.predict(X_test)
    y_prob = (model.predict_proba(X_test)[:, 1]
              if hasattr(model, 'predict_proba') else y_pred)

    recall = recall_score(y_test, y_pred, zero_division=0) * 100
    gmean  = g_mean(y_test, y_pred) * 100
    auc    = roc_auc_score(y_test, y_prob) * 100

    return recall, gmean, auc


# ═════════════════════════════════════════════════════════════
# STEP 7: FIND BEST h
# ═════════════════════════════════════════════════════════════

def find_best_h(clf, X_train, y_train, X_unlabeled, X_test, y_test, clf_name):
    print(f"\n  Finding best h for {clf_name}...")
    best_h, best_score, best_metrics = None, -1, None

    for h in H_VALUES:
        recall, gmean, auc = run_ssstr(
            clf, X_train, y_train,
            X_unlabeled, X_test, y_test,
            h, verbose=False
        )
        # Select best h using balanced metrics only (not Recall alone)
        selection_score = (gmean + auc) / 2
        print(f"  h={h} → Recall={recall:.2f}% | "
              f"G-Mean={gmean:.2f}% | AUC={auc:.2f}% | "
              f"Avg(GMean+AUC)={selection_score:.2f}%")

        if selection_score > best_score:
            best_score   = selection_score
            best_h       = h
            best_metrics = (recall, gmean, auc)

    print(f"\n  ✓ Best h={best_h} (Avg={best_score:.2f}%)")
    return best_h, best_metrics


# ═════════════════════════════════════════════════════════════
# STEP 8: RUN EXPERIMENT PER FEATURE SUBSET
# ═════════════════════════════════════════════════════════════

def run_experiment(X_train, y_train,
                   X_unlabeled, X_test, y_test,
                   norm_name, subset_name, features, fold=None):

    fold_str = f" | Fold {fold}" if fold is not None else ""
    print(f"\n{'='*60}")
    print(f"  Normalization : {norm_name}{fold_str}")
    print(f"  Feature Subset: {subset_name} → {features}")
    print(f"{'='*60}")

    results = []

    X_tr_sub  = X_train[features].reset_index(drop=True)
    X_te_sub  = X_test[features].reset_index(drop=True)
    X_unl_sub = X_unlabeled[features].reset_index(drop=True)
    y_tr_nors = pd.Series(y_train).reset_index(drop=True)

    print(f"\n  Applying resampling on {subset_name} ({len(features)} features)...")
    X_tr_smote, y_tr_smote = apply_smote(X_tr_sub, y_tr_nors)
    X_tr_cbute, y_tr_cbute = apply_cbute(X_tr_sub, y_tr_nors)

    sampling_sets = {
        'NORS' : (X_tr_sub,    y_tr_nors),
        'SMOTE': (X_tr_smote,  y_tr_smote),
        'CBUTE': (X_tr_cbute,  y_tr_cbute)
    }

    for clf_name, clf in classifiers.items():
        for sampling_name, (X_tr, y_tr) in sampling_sets.items():
            print(f"\n{'─'*60}")
            print(f"  Classifier: {clf_name} | Resampling: {sampling_name}")
            print(f"  Train size: {len(X_tr)} | "
                  f"Class dist: {pd.Series(y_tr).value_counts().to_dict()}")
            print(f"{'─'*60}")

            best_h, (recall, gmean, auc) = find_best_h(
                clf,
                X_tr, y_tr,
                X_unl_sub,
                X_te_sub, y_test,
                clf_name
            )

            results.append({
                'Fold'         : fold,
                'Normalization': norm_name,
                'Subset'       : subset_name,
                'Classifier'   : clf_name,
                'Resampling'   : sampling_name,
                'Best_h'       : best_h,
                'Recall'       : round(recall, 2),
                'G-Mean'       : round(gmean, 2),
                'AUC'          : round(auc, 2)
            })

            print(f"\n  ✓ {clf_name} | {sampling_name} | {norm_name} | {subset_name}")
            print(f"    Best h={best_h} | Recall={recall:.2f}% | "
                  f"G-Mean={gmean:.2f}% | AUC={auc:.2f}%")

    return pd.DataFrame(results)


# ═════════════════════════════════════════════════════════════
# STEP 9: RUN SINGLE FOLD ONLY
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print(f"STEP 9: RUNNING FOLD {RUN_ONLY_FOLD} OF 5")
print("="*60)

N_FOLDS = 5
skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
X_all   = labeled[all_features]
y_all   = labeled[target]

fold_results = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all), 1):
    if fold != RUN_ONLY_FOLD:
        continue  # skip all other folds

    print(f"\n{'='*60}")
    print(f"  FOLD {fold}/{N_FOLDS}")
    print(f"{'='*60}")

    X_train_raw = X_all.iloc[train_idx].reset_index(drop=True)
    X_test_raw  = X_all.iloc[test_idx].reset_index(drop=True)
    y_train     = y_all.iloc[train_idx].reset_index(drop=True)
    y_test      = y_all.iloc[test_idx].reset_index(drop=True)

    print(f"  Train: {len(X_train_raw)} | Test: {len(X_test_raw)}")
    print(f"  Train dist: {y_train.value_counts().to_dict()}")
    print(f"  Test dist : {y_test.value_counts().to_dict()}")

    # Normalize per fold (fit on train only)
    normZ         = Normalizer()
    X_train_Z     = normZ.fit_transform(X_train_raw, columns_to_normalize, method='zscore')
    X_test_Z      = normZ.transform(X_test_raw,      columns_to_normalize)
    X_unlabeled_Z = normZ.transform(unlabeled[all_features], columns_to_normalize)

    normM         = Normalizer()
    X_train_M     = normM.fit_transform(X_train_raw, columns_to_normalize, method='minmax')
    X_test_M      = normM.transform(X_test_raw,      columns_to_normalize)
    X_unlabeled_M = normM.transform(unlabeled[all_features], columns_to_normalize)

    experiment_configs = {
        'Z-Score': (X_train_Z, X_test_Z, X_unlabeled_Z),
        'Min-Max': (X_train_M, X_test_M, X_unlabeled_M),
    }

    for subset_name, features in feature_subsets.items():
        for norm_name, (X_tr, X_te, X_unl) in experiment_configs.items():
            results = run_experiment(
                X_tr, y_train,
                X_unl, X_te, y_test,
                norm_name, subset_name, features,
                fold=fold
            )
            fold_results.append(results)

            # Checkpoint save after every combination
            pd.concat(fold_results, ignore_index=True).to_csv(
                OUTPUT_FILE, index=False
            )
            print(f"  ✓ Checkpoint saved: Fold {fold} | {subset_name} | {norm_name}")

# ═════════════════════════════════════════════════════════════
# STEP 10: SAVE FOLD RESULTS
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print(f"STEP 10: FOLD {RUN_ONLY_FOLD} RESULTS")
print("="*60)

final = pd.concat(fold_results, ignore_index=True)
final.to_csv(OUTPUT_FILE, index=False)
print(f"\n✓ Results saved to {OUTPUT_FILE}")
print(final.to_string(index=False))

# ═════════════════════════════════════════════════════════════
# MERGE SCRIPT — run this after ALL 5 folds are complete
# ═════════════════════════════════════════════════════════════
# import pandas as pd, glob
#
# files    = sorted(glob.glob(f'{OUT_PATH}/ssstr_fold_*.csv'))
# combined = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
# combined.to_csv(f'{OUT_PATH}/ssstr_cv_all_folds.csv', index=False)
#
# avg_results = combined.groupby(
#     ['Normalization', 'Subset', 'Classifier', 'Resampling']
# )[['Recall', 'G-Mean', 'AUC']].agg(['mean', 'std']).round(2)
#
# avg_results.columns = ['Recall_mean', 'Recall_std',
#                        'GMean_mean',  'GMean_std',
#                        'AUC_mean',    'AUC_std']
# avg_results = avg_results.reset_index()
# avg_results.to_csv(f'{OUT_PATH}/ssstr_cv_results.csv', index=False)
# print(avg_results.to_string(index=False))


CONFIG
  Running fold     : 3 of 5
  Labeled fraction : 10%
  Add noise        : True (level=0.3)
  Labelled path    : /kaggle/input/datasets/mahsaheidary/labelled-dataset/Labelled.csv
  Unlabelled path  : /kaggle/input/datasets/mahsaheidary/unlabelled-dataset/Unlabelled.csv
  Output file      : /kaggle/working/ssstr_fold_3.csv

STEP 1: LOADING DATA
Full labeled dataset : (6873, 11)
Unlabeled dataset    : (23877, 10)
Full class dist:
Fake
1    4020
0    2853
Name: count, dtype: int64

[Option 1] Using 10% as labeled → 687 rows
  Unlabeled pool now: 30063 rows
  Labeled class dist:
Fake
1    402
0    285
Name: count, dtype: int64

[Option 2] Gaussian noise added to: ['likesCount', 'videoCount', 'followerCount', 'followingCount']
  Noise level: 0.3 × feature std

STEP 2: FEATURE SELECTION (RFE-SVM + BIT-FLIP)

── Running RFE-SVM ──

Feature Rankings (1 = most important):
            Feature  Ranking
Nickname_Complexity        1
         videoCount        2
                Bio        3
 

In [2]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import recall_score, roc_auc_score
from sklearn.cluster import KMeans
from imblearn.over_sampling import SMOTE
import os
import copy
import warnings
warnings.filterwarnings('ignore')

# ═════════════════════════════════════════════════════════════
# CONFIG — change these per session
# ═════════════════════════════════════════════════════════════
RUN_ONLY_FOLD    = 4       # ← SET THIS: 1, 2, 3, 4, or 5
LABELED_FRACTION = 0.10
ADD_NOISE        = True
NOISE_LEVEL      = 0.3
NOISE_COLS       = ['likesCount', 'videoCount', 'followerCount', 'followingCount']
H_VALUES         = [0.1, 0.2, 0.3, 0.5]

# ── Kaggle dataset names ──────────────────────────────────────
# Run this first in a Kaggle cell to confirm exact names:
#   import os; print(os.listdir('/kaggle/input'))
# Then paste the names below:
KAGGLE_DATASET_LABELLED   = 'Labelled.csv'    # ← CHANGE THIS
KAGGLE_DATASET_UNLABELLED = 'Unlabelled.csv'  # ← CHANGE THIS
# ─────────────────────────────────────────────────────────────

LABELLED_PATH   = '/kaggle/input/datasets/mahsaheidary/labelled-dataset/Labelled.csv'
UNLABELLED_PATH = '/kaggle/input/datasets/mahsaheidary/unlabelled-dataset/Unlabelled.csv'
OUT_PATH        = '/kaggle/working'

OUTPUT_FILE = f'{OUT_PATH}/ssstr_fold_{RUN_ONLY_FOLD}.csv'

print("\n" + "="*60)
print("CONFIG")
print("="*60)
print(f"  Running fold     : {RUN_ONLY_FOLD} of 5")
print(f"  Labeled fraction : {LABELED_FRACTION*100:.0f}%")
print(f"  Add noise        : {ADD_NOISE} (level={NOISE_LEVEL})")
print(f"  Labelled path    : {LABELLED_PATH}")
print(f"  Unlabelled path  : {UNLABELLED_PATH}")
print(f"  Output file      : {OUTPUT_FILE}")
print("="*60)

# ═════════════════════════════════════════════════════════════
# STEP 1: LOAD DATA
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 1: LOADING DATA")
print("="*60)

labeled_full = pd.read_csv(LABELLED_PATH)
unlabeled    = pd.read_csv(UNLABELLED_PATH)

all_features = [
    'likesCount', 'videoCount', 'followerCount', 'followingCount',
    'verified', 'Bio', 'Has_Contact_Info', 'Jaro_Similarity', 'Nickname_Complexity'
]
target = 'Fake'

print(f"Full labeled dataset : {labeled_full.shape}")
print(f"Unlabeled dataset    : {unlabeled.shape}")
print(f"Full class dist:\n{labeled_full[target].value_counts()}")

# ─────────────────────────────────────────────────────────────
# OPTION 1: Reduce labeled pool
# The held-out portion is moved into the unlabeled pool so
# the model can still leverage it via SSSTR.
# ─────────────────────────────────────────────────────────────
if LABELED_FRACTION < 1.0:
    labeled, extra_unlabeled = train_test_split(
        labeled_full,
        train_size=LABELED_FRACTION,
        stratify=labeled_full[target],
        random_state=42
    )
    labeled          = labeled.reset_index(drop=True)
    extra_unlabeled  = extra_unlabeled[all_features].reset_index(drop=True)
    # Append held-out rows (without labels) to unlabeled pool
    unlabeled        = pd.concat(
        [unlabeled[all_features], extra_unlabeled], ignore_index=True
    )
    print(f"\n[Option 1] Using {LABELED_FRACTION*100:.0f}% as labeled → {len(labeled)} rows")
    print(f"  Unlabeled pool now: {len(unlabeled)} rows")
    print(f"  Labeled class dist:\n{labeled[target].value_counts()}")
else:
    labeled = labeled_full.copy()
    print("\n[Option 1] Using 100% of labeled data (no reduction)")

# ─────────────────────────────────────────────────────────────
# OPTION 2: Add Gaussian noise to continuous features
# ─────────────────────────────────────────────────────────────
if ADD_NOISE:
    np.random.seed(42)
    for col in NOISE_COLS:
        std = labeled[col].std()
        labeled[col]   = labeled[col]   + np.random.normal(0, std * NOISE_LEVEL, size=len(labeled))
        unlabeled[col] = unlabeled[col] + np.random.normal(0, std * NOISE_LEVEL, size=len(unlabeled))
    print(f"\n[Option 2] Gaussian noise added to: {NOISE_COLS}")
    print(f"  Noise level: {NOISE_LEVEL} × feature std")
else:
    print("\n[Option 2] No noise added")

# ═════════════════════════════════════════════════════════════
# STEP 2: RFE-SVM FEATURE SELECTION + BIT-FLIP
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 2: FEATURE SELECTION (RFE-SVM + BIT-FLIP)")
print("="*60)

X_fs = labeled[all_features]
y_fs = labeled[target]

scaler_fs   = StandardScaler()
X_fs_scaled = pd.DataFrame(scaler_fs.fit_transform(X_fs), columns=all_features)

# --- RFE-SVM ---
print("\n── Running RFE-SVM ──")
svm = SVC(kernel='linear', random_state=42)
rfe = RFE(estimator=svm, n_features_to_select=1, step=1)
rfe.fit(X_fs_scaled, y_fs)

feature_ranking = pd.DataFrame({
    'Feature': all_features,
    'Ranking': rfe.ranking_
}).sort_values('Ranking')

print("\nFeature Rankings (1 = most important):")
print(feature_ranking.to_string(index=False))

ranked_features = feature_ranking['Feature'].tolist()
subset_3 = ranked_features[:3]
subset_5 = ranked_features[:5]
subset_7 = ranked_features[:7]

print(f"\n3-feature subset: {subset_3}")
print(f"5-feature subset: {subset_5}")
print(f"7-feature subset: {subset_7}")


# --- Bit-Flip Local Search ---
def evaluate_subset(features, X, y):
    """Evaluates feature subset using 5-fold stratified cross-validation."""
    if len(features) == 0:
        return 0.0
    clf = DecisionTreeClassifier(random_state=42)
    cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in cv.split(X[features], y):
        X_tr, X_val = X[features].iloc[train_idx], X[features].iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        sc = StandardScaler()
        X_tr_sc  = sc.fit_transform(X_tr)
        X_val_sc = sc.transform(X_val)
        clf_cv = copy.deepcopy(clf)
        clf_cv.fit(X_tr_sc, y_tr)
        scores.append(clf_cv.score(X_val_sc, y_val))
    return np.mean(scores)


def bit_flip_search(initial_subset, all_features, X, y):
    """Adds or removes one feature at a time to find better subset."""
    print("\n── Running Bit-Flip Local Search ──")
    print(f"  Starting subset  : {initial_subset}")

    current_subset = initial_subset.copy()
    current_score  = evaluate_subset(current_subset, X, y)
    print(f"  Starting accuracy: {current_score:.4f}")

    improved  = True
    iteration = 0

    while improved:
        improved  = False
        iteration += 1
        print(f"\n  [Iteration {iteration}]")
        best_subset = current_subset.copy()
        best_score  = current_score

        print("  Testing removals...")
        for feature in current_subset:
            candidate = [f for f in current_subset if f != feature]
            if len(candidate) == 0:
                continue
            score = evaluate_subset(candidate, X, y)
            print(f"    Remove '{feature}': {score:.4f}")
            if score > best_score:
                best_score  = score
                best_subset = candidate
                improved    = True

        print("  Testing additions...")
        for feature in [f for f in all_features if f not in current_subset]:
            candidate = current_subset + [feature]
            score     = evaluate_subset(candidate, X, y)
            print(f"    Add '{feature}': {score:.4f}")
            if score > best_score:
                best_score  = score
                best_subset = candidate
                improved    = True

        if improved:
            current_subset = best_subset
            current_score  = best_score
            print(f"\n  ✓ Improved → {current_subset} (accuracy={current_score:.4f})")
        else:
            print(f"\n  ✓ No improvement. Stopping.")

    print(f"\n  Best subset  : {current_subset}")
    print(f"  Best accuracy: {current_score:.4f}")
    return current_subset, current_score


best_subset, best_score = bit_flip_search(subset_5, all_features, X_fs, y_fs)

feature_subsets = {
    '3_features'   : subset_3,
    '5_features'   : subset_5,
    '7_features'   : subset_7,
    'best_features': best_subset
}

print("\n" + "="*60)
print("FEATURE SUBSETS SUMMARY")
print("="*60)
for name, feats in feature_subsets.items():
    print(f"  {name:15s}: {feats}")

# ═════════════════════════════════════════════════════════════
# STEP 3: NORMALIZER CLASS
# ═════════════════════════════════════════════════════════════
columns_to_normalize = [
    'likesCount', 'videoCount', 'followerCount', 'followingCount',
    'Jaro_Similarity', 'Nickname_Complexity'
]


class Normalizer:
    def __init__(self):
        self.scaler = None

    def fit_transform(self, df, columns, method='zscore'):
        missing = [c for c in columns if c not in df.columns]
        if missing:
            raise ValueError(f"Columns not found: {missing}")
        if method == 'zscore':
            self.scaler = StandardScaler()
        elif method == 'minmax':
            self.scaler = MinMaxScaler()
        else:
            raise ValueError(f"Unknown method: {method}")
        out          = df.copy()
        out[columns] = self.scaler.fit_transform(df[columns])
        return out

    def transform(self, df, columns):
        if self.scaler is None:
            raise ValueError("Scaler not fitted yet.")
        missing = [c for c in columns if c not in df.columns]
        if missing:
            raise ValueError(f"Columns not found: {missing}")
        out          = df.copy()
        out[columns] = self.scaler.transform(df[columns])
        return out


# ═════════════════════════════════════════════════════════════
# STEP 4: RESAMPLING FUNCTIONS
# ═════════════════════════════════════════════════════════════

def apply_smote(X_train, y_train, random_state=42):
    y_train = pd.Series(y_train).reset_index(drop=True)
    minority_count = y_train.value_counts().min()
    k = min(5, minority_count - 1)
    if k < 1:
        print("  SMOTE skipped: minority class too small")
        return X_train, y_train
    smote        = SMOTE(random_state=random_state, k_neighbors=k)
    X_res, y_res = smote.fit_resample(X_train, y_train)
    print(f"  After SMOTE: {pd.Series(y_res).value_counts().to_dict()}")
    return X_res, pd.Series(y_res).reset_index(drop=True)


def apply_cbute(X_train, y_train, random_state=42):
    X_train = X_train.reset_index(drop=True)
    y_train = pd.Series(y_train).reset_index(drop=True)

    minority_class = y_train.value_counts().idxmin()
    majority_class = y_train.value_counts().idxmax()

    X_min = X_train[y_train == minority_class]
    X_maj = X_train[y_train == majority_class]
    y_min = y_train[y_train == minority_class]

    n_clusters = max(1, len(X_min))
    kmeans     = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    kmeans.fit(X_maj)

    X_maj_reduced = pd.DataFrame(kmeans.cluster_centers_, columns=X_train.columns)
    y_maj_reduced = pd.Series([majority_class] * n_clusters)

    X_res = pd.concat([X_min.reset_index(drop=True), X_maj_reduced], ignore_index=True)
    y_res = pd.Series(
        pd.concat([y_min.reset_index(drop=True), y_maj_reduced], ignore_index=True)
    ).reset_index(drop=True)

    print(f"  After CBUTE: {y_res.value_counts().to_dict()}")
    return X_res, y_res


# ═════════════════════════════════════════════════════════════
# STEP 5: CLASSIFIERS + METRICS
# ═════════════════════════════════════════════════════════════

classifiers = {
    'CART': DecisionTreeClassifier(random_state=42),
    'RF'  : RandomForestClassifier(n_estimators=20, random_state=42),
    'GB'  : GradientBoostingClassifier(n_estimators=20, random_state=42),
    'AB'  : AdaBoostClassifier(random_state=42, algorithm='SAMME'),
    'KNN' : KNeighborsClassifier(),
    'NB'  : GaussianNB()
}


def g_mean(y_true, y_pred):
    recall = recall_score(y_true, y_pred, zero_division=0)
    tnr    = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    return np.sqrt(recall * tnr)


# ═════════════════════════════════════════════════════════════
# STEP 6: SSSTR ALGORITHM
# ═════════════════════════════════════════════════════════════

def run_ssstr(clf, X_train, y_train, X_unlabeled, X_test, y_test, h, verbose=False):
    L_X = X_train.copy().reset_index(drop=True)
    L_y = pd.Series(y_train).copy().reset_index(drop=True)
    U_X = X_unlabeled.copy().reset_index(drop=True)

    iteration = 1
    while len(U_X) > 0:
        model = copy.deepcopy(clf)
        model.fit(L_X, L_y)

        if hasattr(model, 'predict_proba'):
            probs        = model.predict_proba(U_X)
            confidence   = probs.max(axis=1)
            pred_col_idx = probs.argmax(axis=1)
            preds        = np.array([model.classes_[i] for i in pred_col_idx])
        else:
            preds      = model.predict(U_X)
            confidence = np.ones(len(preds))

        n_select = max(1, int(h * len(U_X)))
        top_idx  = np.argsort(confidence)[::-1][:n_select]

        L_X = pd.concat([L_X, U_X.iloc[top_idx]],        ignore_index=True)
        L_y = pd.concat([L_y, pd.Series(preds[top_idx])], ignore_index=True)
        U_X = U_X.drop(index=U_X.index[top_idx]).reset_index(drop=True)

        iteration += 1

    model = copy.deepcopy(clf)
    model.fit(L_X, L_y)

    y_pred = model.predict(X_test)
    y_prob = (model.predict_proba(X_test)[:, 1]
              if hasattr(model, 'predict_proba') else y_pred)

    recall = recall_score(y_test, y_pred, zero_division=0) * 100
    gmean  = g_mean(y_test, y_pred) * 100
    auc    = roc_auc_score(y_test, y_prob) * 100

    return recall, gmean, auc


# ═════════════════════════════════════════════════════════════
# STEP 7: FIND BEST h
# ═════════════════════════════════════════════════════════════

def find_best_h(clf, X_train, y_train, X_unlabeled, X_test, y_test, clf_name):
    print(f"\n  Finding best h for {clf_name}...")
    best_h, best_score, best_metrics = None, -1, None

    for h in H_VALUES:
        recall, gmean, auc = run_ssstr(
            clf, X_train, y_train,
            X_unlabeled, X_test, y_test,
            h, verbose=False
        )
        # Select best h using balanced metrics only (not Recall alone)
        selection_score = (gmean + auc) / 2
        print(f"  h={h} → Recall={recall:.2f}% | "
              f"G-Mean={gmean:.2f}% | AUC={auc:.2f}% | "
              f"Avg(GMean+AUC)={selection_score:.2f}%")

        if selection_score > best_score:
            best_score   = selection_score
            best_h       = h
            best_metrics = (recall, gmean, auc)

    print(f"\n  ✓ Best h={best_h} (Avg={best_score:.2f}%)")
    return best_h, best_metrics


# ═════════════════════════════════════════════════════════════
# STEP 8: RUN EXPERIMENT PER FEATURE SUBSET
# ═════════════════════════════════════════════════════════════

def run_experiment(X_train, y_train,
                   X_unlabeled, X_test, y_test,
                   norm_name, subset_name, features, fold=None):

    fold_str = f" | Fold {fold}" if fold is not None else ""
    print(f"\n{'='*60}")
    print(f"  Normalization : {norm_name}{fold_str}")
    print(f"  Feature Subset: {subset_name} → {features}")
    print(f"{'='*60}")

    results = []

    X_tr_sub  = X_train[features].reset_index(drop=True)
    X_te_sub  = X_test[features].reset_index(drop=True)
    X_unl_sub = X_unlabeled[features].reset_index(drop=True)
    y_tr_nors = pd.Series(y_train).reset_index(drop=True)

    print(f"\n  Applying resampling on {subset_name} ({len(features)} features)...")
    X_tr_smote, y_tr_smote = apply_smote(X_tr_sub, y_tr_nors)
    X_tr_cbute, y_tr_cbute = apply_cbute(X_tr_sub, y_tr_nors)

    sampling_sets = {
        'NORS' : (X_tr_sub,    y_tr_nors),
        'SMOTE': (X_tr_smote,  y_tr_smote),
        'CBUTE': (X_tr_cbute,  y_tr_cbute)
    }

    for clf_name, clf in classifiers.items():
        for sampling_name, (X_tr, y_tr) in sampling_sets.items():
            print(f"\n{'─'*60}")
            print(f"  Classifier: {clf_name} | Resampling: {sampling_name}")
            print(f"  Train size: {len(X_tr)} | "
                  f"Class dist: {pd.Series(y_tr).value_counts().to_dict()}")
            print(f"{'─'*60}")

            best_h, (recall, gmean, auc) = find_best_h(
                clf,
                X_tr, y_tr,
                X_unl_sub,
                X_te_sub, y_test,
                clf_name
            )

            results.append({
                'Fold'         : fold,
                'Normalization': norm_name,
                'Subset'       : subset_name,
                'Classifier'   : clf_name,
                'Resampling'   : sampling_name,
                'Best_h'       : best_h,
                'Recall'       : round(recall, 2),
                'G-Mean'       : round(gmean, 2),
                'AUC'          : round(auc, 2)
            })

            print(f"\n  ✓ {clf_name} | {sampling_name} | {norm_name} | {subset_name}")
            print(f"    Best h={best_h} | Recall={recall:.2f}% | "
                  f"G-Mean={gmean:.2f}% | AUC={auc:.2f}%")

    return pd.DataFrame(results)


# ═════════════════════════════════════════════════════════════
# STEP 9: RUN SINGLE FOLD ONLY
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print(f"STEP 9: RUNNING FOLD {RUN_ONLY_FOLD} OF 5")
print("="*60)

N_FOLDS = 5
skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
X_all   = labeled[all_features]
y_all   = labeled[target]

fold_results = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all), 1):
    if fold != RUN_ONLY_FOLD:
        continue  # skip all other folds

    print(f"\n{'='*60}")
    print(f"  FOLD {fold}/{N_FOLDS}")
    print(f"{'='*60}")

    X_train_raw = X_all.iloc[train_idx].reset_index(drop=True)
    X_test_raw  = X_all.iloc[test_idx].reset_index(drop=True)
    y_train     = y_all.iloc[train_idx].reset_index(drop=True)
    y_test      = y_all.iloc[test_idx].reset_index(drop=True)

    print(f"  Train: {len(X_train_raw)} | Test: {len(X_test_raw)}")
    print(f"  Train dist: {y_train.value_counts().to_dict()}")
    print(f"  Test dist : {y_test.value_counts().to_dict()}")

    # Normalize per fold (fit on train only)
    normZ         = Normalizer()
    X_train_Z     = normZ.fit_transform(X_train_raw, columns_to_normalize, method='zscore')
    X_test_Z      = normZ.transform(X_test_raw,      columns_to_normalize)
    X_unlabeled_Z = normZ.transform(unlabeled[all_features], columns_to_normalize)

    normM         = Normalizer()
    X_train_M     = normM.fit_transform(X_train_raw, columns_to_normalize, method='minmax')
    X_test_M      = normM.transform(X_test_raw,      columns_to_normalize)
    X_unlabeled_M = normM.transform(unlabeled[all_features], columns_to_normalize)

    experiment_configs = {
        'Z-Score': (X_train_Z, X_test_Z, X_unlabeled_Z),
        'Min-Max': (X_train_M, X_test_M, X_unlabeled_M),
    }

    for subset_name, features in feature_subsets.items():
        for norm_name, (X_tr, X_te, X_unl) in experiment_configs.items():
            results = run_experiment(
                X_tr, y_train,
                X_unl, X_te, y_test,
                norm_name, subset_name, features,
                fold=fold
            )
            fold_results.append(results)

            # Checkpoint save after every combination
            pd.concat(fold_results, ignore_index=True).to_csv(
                OUTPUT_FILE, index=False
            )
            print(f"  ✓ Checkpoint saved: Fold {fold} | {subset_name} | {norm_name}")

# ═════════════════════════════════════════════════════════════
# STEP 10: SAVE FOLD RESULTS
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print(f"STEP 10: FOLD {RUN_ONLY_FOLD} RESULTS")
print("="*60)

final = pd.concat(fold_results, ignore_index=True)
final.to_csv(OUTPUT_FILE, index=False)
print(f"\n✓ Results saved to {OUTPUT_FILE}")
print(final.to_string(index=False))

# ═════════════════════════════════════════════════════════════
# MERGE SCRIPT — run this after ALL 5 folds are complete
# ═════════════════════════════════════════════════════════════
# import pandas as pd, glob
#
# files    = sorted(glob.glob(f'{OUT_PATH}/ssstr_fold_*.csv'))
# combined = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
# combined.to_csv(f'{OUT_PATH}/ssstr_cv_all_folds.csv', index=False)
#
# avg_results = combined.groupby(
#     ['Normalization', 'Subset', 'Classifier', 'Resampling']
# )[['Recall', 'G-Mean', 'AUC']].agg(['mean', 'std']).round(2)
#
# avg_results.columns = ['Recall_mean', 'Recall_std',
#                        'GMean_mean',  'GMean_std',
#                        'AUC_mean',    'AUC_std']
# avg_results = avg_results.reset_index()
# avg_results.to_csv(f'{OUT_PATH}/ssstr_cv_results.csv', index=False)
# print(avg_results.to_string(index=False))


CONFIG
  Running fold     : 4 of 5
  Labeled fraction : 10%
  Add noise        : True (level=0.3)
  Labelled path    : /kaggle/input/datasets/mahsaheidary/labelled-dataset/Labelled.csv
  Unlabelled path  : /kaggle/input/datasets/mahsaheidary/unlabelled-dataset/Unlabelled.csv
  Output file      : /kaggle/working/ssstr_fold_4.csv

STEP 1: LOADING DATA
Full labeled dataset : (6873, 11)
Unlabeled dataset    : (23877, 10)
Full class dist:
Fake
1    4020
0    2853
Name: count, dtype: int64

[Option 1] Using 10% as labeled → 687 rows
  Unlabeled pool now: 30063 rows
  Labeled class dist:
Fake
1    402
0    285
Name: count, dtype: int64

[Option 2] Gaussian noise added to: ['likesCount', 'videoCount', 'followerCount', 'followingCount']
  Noise level: 0.3 × feature std

STEP 2: FEATURE SELECTION (RFE-SVM + BIT-FLIP)

── Running RFE-SVM ──

Feature Rankings (1 = most important):
            Feature  Ranking
Nickname_Complexity        1
         videoCount        2
                Bio        3
 

In [3]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import recall_score, roc_auc_score
from sklearn.cluster import KMeans
from imblearn.over_sampling import SMOTE
import os
import copy
import warnings
warnings.filterwarnings('ignore')

# ═════════════════════════════════════════════════════════════
# CONFIG — change these per session
# ═════════════════════════════════════════════════════════════
RUN_ONLY_FOLD    = 5       # ← SET THIS: 1, 2, 3, 4, or 5
LABELED_FRACTION = 0.10
ADD_NOISE        = True
NOISE_LEVEL      = 0.3
NOISE_COLS       = ['likesCount', 'videoCount', 'followerCount', 'followingCount']
H_VALUES         = [0.1, 0.2, 0.3, 0.5]

# ── Kaggle dataset names ──────────────────────────────────────
# Run this first in a Kaggle cell to confirm exact names:
#   import os; print(os.listdir('/kaggle/input'))
# Then paste the names below:
KAGGLE_DATASET_LABELLED   = 'Labelled.csv'    # ← CHANGE THIS
KAGGLE_DATASET_UNLABELLED = 'Unlabelled.csv'  # ← CHANGE THIS
# ─────────────────────────────────────────────────────────────

LABELLED_PATH   = '/kaggle/input/datasets/mahsaheidary/labelled-dataset/Labelled.csv'
UNLABELLED_PATH = '/kaggle/input/datasets/mahsaheidary/unlabelled-dataset/Unlabelled.csv'
OUT_PATH        = '/kaggle/working'

OUTPUT_FILE = f'{OUT_PATH}/ssstr_fold_{RUN_ONLY_FOLD}.csv'

print("\n" + "="*60)
print("CONFIG")
print("="*60)
print(f"  Running fold     : {RUN_ONLY_FOLD} of 5")
print(f"  Labeled fraction : {LABELED_FRACTION*100:.0f}%")
print(f"  Add noise        : {ADD_NOISE} (level={NOISE_LEVEL})")
print(f"  Labelled path    : {LABELLED_PATH}")
print(f"  Unlabelled path  : {UNLABELLED_PATH}")
print(f"  Output file      : {OUTPUT_FILE}")
print("="*60)

# ═════════════════════════════════════════════════════════════
# STEP 1: LOAD DATA
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 1: LOADING DATA")
print("="*60)

labeled_full = pd.read_csv(LABELLED_PATH)
unlabeled    = pd.read_csv(UNLABELLED_PATH)

all_features = [
    'likesCount', 'videoCount', 'followerCount', 'followingCount',
    'verified', 'Bio', 'Has_Contact_Info', 'Jaro_Similarity', 'Nickname_Complexity'
]
target = 'Fake'

print(f"Full labeled dataset : {labeled_full.shape}")
print(f"Unlabeled dataset    : {unlabeled.shape}")
print(f"Full class dist:\n{labeled_full[target].value_counts()}")

# ─────────────────────────────────────────────────────────────
# OPTION 1: Reduce labeled pool
# The held-out portion is moved into the unlabeled pool so
# the model can still leverage it via SSSTR.
# ─────────────────────────────────────────────────────────────
if LABELED_FRACTION < 1.0:
    labeled, extra_unlabeled = train_test_split(
        labeled_full,
        train_size=LABELED_FRACTION,
        stratify=labeled_full[target],
        random_state=42
    )
    labeled          = labeled.reset_index(drop=True)
    extra_unlabeled  = extra_unlabeled[all_features].reset_index(drop=True)
    # Append held-out rows (without labels) to unlabeled pool
    unlabeled        = pd.concat(
        [unlabeled[all_features], extra_unlabeled], ignore_index=True
    )
    print(f"\n[Option 1] Using {LABELED_FRACTION*100:.0f}% as labeled → {len(labeled)} rows")
    print(f"  Unlabeled pool now: {len(unlabeled)} rows")
    print(f"  Labeled class dist:\n{labeled[target].value_counts()}")
else:
    labeled = labeled_full.copy()
    print("\n[Option 1] Using 100% of labeled data (no reduction)")

# ─────────────────────────────────────────────────────────────
# OPTION 2: Add Gaussian noise to continuous features
# ─────────────────────────────────────────────────────────────
if ADD_NOISE:
    np.random.seed(42)
    for col in NOISE_COLS:
        std = labeled[col].std()
        labeled[col]   = labeled[col]   + np.random.normal(0, std * NOISE_LEVEL, size=len(labeled))
        unlabeled[col] = unlabeled[col] + np.random.normal(0, std * NOISE_LEVEL, size=len(unlabeled))
    print(f"\n[Option 2] Gaussian noise added to: {NOISE_COLS}")
    print(f"  Noise level: {NOISE_LEVEL} × feature std")
else:
    print("\n[Option 2] No noise added")

# ═════════════════════════════════════════════════════════════
# STEP 2: RFE-SVM FEATURE SELECTION + BIT-FLIP
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 2: FEATURE SELECTION (RFE-SVM + BIT-FLIP)")
print("="*60)

X_fs = labeled[all_features]
y_fs = labeled[target]

scaler_fs   = StandardScaler()
X_fs_scaled = pd.DataFrame(scaler_fs.fit_transform(X_fs), columns=all_features)

# --- RFE-SVM ---
print("\n── Running RFE-SVM ──")
svm = SVC(kernel='linear', random_state=42)
rfe = RFE(estimator=svm, n_features_to_select=1, step=1)
rfe.fit(X_fs_scaled, y_fs)

feature_ranking = pd.DataFrame({
    'Feature': all_features,
    'Ranking': rfe.ranking_
}).sort_values('Ranking')

print("\nFeature Rankings (1 = most important):")
print(feature_ranking.to_string(index=False))

ranked_features = feature_ranking['Feature'].tolist()
subset_3 = ranked_features[:3]
subset_5 = ranked_features[:5]
subset_7 = ranked_features[:7]

print(f"\n3-feature subset: {subset_3}")
print(f"5-feature subset: {subset_5}")
print(f"7-feature subset: {subset_7}")


# --- Bit-Flip Local Search ---
def evaluate_subset(features, X, y):
    """Evaluates feature subset using 5-fold stratified cross-validation."""
    if len(features) == 0:
        return 0.0
    clf = DecisionTreeClassifier(random_state=42)
    cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in cv.split(X[features], y):
        X_tr, X_val = X[features].iloc[train_idx], X[features].iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        sc = StandardScaler()
        X_tr_sc  = sc.fit_transform(X_tr)
        X_val_sc = sc.transform(X_val)
        clf_cv = copy.deepcopy(clf)
        clf_cv.fit(X_tr_sc, y_tr)
        scores.append(clf_cv.score(X_val_sc, y_val))
    return np.mean(scores)


def bit_flip_search(initial_subset, all_features, X, y):
    """Adds or removes one feature at a time to find better subset."""
    print("\n── Running Bit-Flip Local Search ──")
    print(f"  Starting subset  : {initial_subset}")

    current_subset = initial_subset.copy()
    current_score  = evaluate_subset(current_subset, X, y)
    print(f"  Starting accuracy: {current_score:.4f}")

    improved  = True
    iteration = 0

    while improved:
        improved  = False
        iteration += 1
        print(f"\n  [Iteration {iteration}]")
        best_subset = current_subset.copy()
        best_score  = current_score

        print("  Testing removals...")
        for feature in current_subset:
            candidate = [f for f in current_subset if f != feature]
            if len(candidate) == 0:
                continue
            score = evaluate_subset(candidate, X, y)
            print(f"    Remove '{feature}': {score:.4f}")
            if score > best_score:
                best_score  = score
                best_subset = candidate
                improved    = True

        print("  Testing additions...")
        for feature in [f for f in all_features if f not in current_subset]:
            candidate = current_subset + [feature]
            score     = evaluate_subset(candidate, X, y)
            print(f"    Add '{feature}': {score:.4f}")
            if score > best_score:
                best_score  = score
                best_subset = candidate
                improved    = True

        if improved:
            current_subset = best_subset
            current_score  = best_score
            print(f"\n  ✓ Improved → {current_subset} (accuracy={current_score:.4f})")
        else:
            print(f"\n  ✓ No improvement. Stopping.")

    print(f"\n  Best subset  : {current_subset}")
    print(f"  Best accuracy: {current_score:.4f}")
    return current_subset, current_score


best_subset, best_score = bit_flip_search(subset_5, all_features, X_fs, y_fs)

feature_subsets = {
    '3_features'   : subset_3,
    '5_features'   : subset_5,
    '7_features'   : subset_7,
    'best_features': best_subset
}

print("\n" + "="*60)
print("FEATURE SUBSETS SUMMARY")
print("="*60)
for name, feats in feature_subsets.items():
    print(f"  {name:15s}: {feats}")

# ═════════════════════════════════════════════════════════════
# STEP 3: NORMALIZER CLASS
# ═════════════════════════════════════════════════════════════
columns_to_normalize = [
    'likesCount', 'videoCount', 'followerCount', 'followingCount',
    'Jaro_Similarity', 'Nickname_Complexity'
]


class Normalizer:
    def __init__(self):
        self.scaler = None

    def fit_transform(self, df, columns, method='zscore'):
        missing = [c for c in columns if c not in df.columns]
        if missing:
            raise ValueError(f"Columns not found: {missing}")
        if method == 'zscore':
            self.scaler = StandardScaler()
        elif method == 'minmax':
            self.scaler = MinMaxScaler()
        else:
            raise ValueError(f"Unknown method: {method}")
        out          = df.copy()
        out[columns] = self.scaler.fit_transform(df[columns])
        return out

    def transform(self, df, columns):
        if self.scaler is None:
            raise ValueError("Scaler not fitted yet.")
        missing = [c for c in columns if c not in df.columns]
        if missing:
            raise ValueError(f"Columns not found: {missing}")
        out          = df.copy()
        out[columns] = self.scaler.transform(df[columns])
        return out


# ═════════════════════════════════════════════════════════════
# STEP 4: RESAMPLING FUNCTIONS
# ═════════════════════════════════════════════════════════════

def apply_smote(X_train, y_train, random_state=42):
    y_train = pd.Series(y_train).reset_index(drop=True)
    minority_count = y_train.value_counts().min()
    k = min(5, minority_count - 1)
    if k < 1:
        print("  SMOTE skipped: minority class too small")
        return X_train, y_train
    smote        = SMOTE(random_state=random_state, k_neighbors=k)
    X_res, y_res = smote.fit_resample(X_train, y_train)
    print(f"  After SMOTE: {pd.Series(y_res).value_counts().to_dict()}")
    return X_res, pd.Series(y_res).reset_index(drop=True)


def apply_cbute(X_train, y_train, random_state=42):
    X_train = X_train.reset_index(drop=True)
    y_train = pd.Series(y_train).reset_index(drop=True)

    minority_class = y_train.value_counts().idxmin()
    majority_class = y_train.value_counts().idxmax()

    X_min = X_train[y_train == minority_class]
    X_maj = X_train[y_train == majority_class]
    y_min = y_train[y_train == minority_class]

    n_clusters = max(1, len(X_min))
    kmeans     = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    kmeans.fit(X_maj)

    X_maj_reduced = pd.DataFrame(kmeans.cluster_centers_, columns=X_train.columns)
    y_maj_reduced = pd.Series([majority_class] * n_clusters)

    X_res = pd.concat([X_min.reset_index(drop=True), X_maj_reduced], ignore_index=True)
    y_res = pd.Series(
        pd.concat([y_min.reset_index(drop=True), y_maj_reduced], ignore_index=True)
    ).reset_index(drop=True)

    print(f"  After CBUTE: {y_res.value_counts().to_dict()}")
    return X_res, y_res


# ═════════════════════════════════════════════════════════════
# STEP 5: CLASSIFIERS + METRICS
# ═════════════════════════════════════════════════════════════

classifiers = {
    'CART': DecisionTreeClassifier(random_state=42),
    'RF'  : RandomForestClassifier(n_estimators=20, random_state=42),
    'GB'  : GradientBoostingClassifier(n_estimators=20, random_state=42),
    'AB'  : AdaBoostClassifier(random_state=42, algorithm='SAMME'),
    'KNN' : KNeighborsClassifier(),
    'NB'  : GaussianNB()
}


def g_mean(y_true, y_pred):
    recall = recall_score(y_true, y_pred, zero_division=0)
    tnr    = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    return np.sqrt(recall * tnr)


# ═════════════════════════════════════════════════════════════
# STEP 6: SSSTR ALGORITHM
# ═════════════════════════════════════════════════════════════

def run_ssstr(clf, X_train, y_train, X_unlabeled, X_test, y_test, h, verbose=False):
    L_X = X_train.copy().reset_index(drop=True)
    L_y = pd.Series(y_train).copy().reset_index(drop=True)
    U_X = X_unlabeled.copy().reset_index(drop=True)

    iteration = 1
    while len(U_X) > 0:
        model = copy.deepcopy(clf)
        model.fit(L_X, L_y)

        if hasattr(model, 'predict_proba'):
            probs        = model.predict_proba(U_X)
            confidence   = probs.max(axis=1)
            pred_col_idx = probs.argmax(axis=1)
            preds        = np.array([model.classes_[i] for i in pred_col_idx])
        else:
            preds      = model.predict(U_X)
            confidence = np.ones(len(preds))

        n_select = max(1, int(h * len(U_X)))
        top_idx  = np.argsort(confidence)[::-1][:n_select]

        L_X = pd.concat([L_X, U_X.iloc[top_idx]],        ignore_index=True)
        L_y = pd.concat([L_y, pd.Series(preds[top_idx])], ignore_index=True)
        U_X = U_X.drop(index=U_X.index[top_idx]).reset_index(drop=True)

        iteration += 1

    model = copy.deepcopy(clf)
    model.fit(L_X, L_y)

    y_pred = model.predict(X_test)
    y_prob = (model.predict_proba(X_test)[:, 1]
              if hasattr(model, 'predict_proba') else y_pred)

    recall = recall_score(y_test, y_pred, zero_division=0) * 100
    gmean  = g_mean(y_test, y_pred) * 100
    auc    = roc_auc_score(y_test, y_prob) * 100

    return recall, gmean, auc


# ═════════════════════════════════════════════════════════════
# STEP 7: FIND BEST h
# ═════════════════════════════════════════════════════════════

def find_best_h(clf, X_train, y_train, X_unlabeled, X_test, y_test, clf_name):
    print(f"\n  Finding best h for {clf_name}...")
    best_h, best_score, best_metrics = None, -1, None

    for h in H_VALUES:
        recall, gmean, auc = run_ssstr(
            clf, X_train, y_train,
            X_unlabeled, X_test, y_test,
            h, verbose=False
        )
        # Select best h using balanced metrics only (not Recall alone)
        selection_score = (gmean + auc) / 2
        print(f"  h={h} → Recall={recall:.2f}% | "
              f"G-Mean={gmean:.2f}% | AUC={auc:.2f}% | "
              f"Avg(GMean+AUC)={selection_score:.2f}%")

        if selection_score > best_score:
            best_score   = selection_score
            best_h       = h
            best_metrics = (recall, gmean, auc)

    print(f"\n  ✓ Best h={best_h} (Avg={best_score:.2f}%)")
    return best_h, best_metrics


# ═════════════════════════════════════════════════════════════
# STEP 8: RUN EXPERIMENT PER FEATURE SUBSET
# ═════════════════════════════════════════════════════════════

def run_experiment(X_train, y_train,
                   X_unlabeled, X_test, y_test,
                   norm_name, subset_name, features, fold=None):

    fold_str = f" | Fold {fold}" if fold is not None else ""
    print(f"\n{'='*60}")
    print(f"  Normalization : {norm_name}{fold_str}")
    print(f"  Feature Subset: {subset_name} → {features}")
    print(f"{'='*60}")

    results = []

    X_tr_sub  = X_train[features].reset_index(drop=True)
    X_te_sub  = X_test[features].reset_index(drop=True)
    X_unl_sub = X_unlabeled[features].reset_index(drop=True)
    y_tr_nors = pd.Series(y_train).reset_index(drop=True)

    print(f"\n  Applying resampling on {subset_name} ({len(features)} features)...")
    X_tr_smote, y_tr_smote = apply_smote(X_tr_sub, y_tr_nors)
    X_tr_cbute, y_tr_cbute = apply_cbute(X_tr_sub, y_tr_nors)

    sampling_sets = {
        'NORS' : (X_tr_sub,    y_tr_nors),
        'SMOTE': (X_tr_smote,  y_tr_smote),
        'CBUTE': (X_tr_cbute,  y_tr_cbute)
    }

    for clf_name, clf in classifiers.items():
        for sampling_name, (X_tr, y_tr) in sampling_sets.items():
            print(f"\n{'─'*60}")
            print(f"  Classifier: {clf_name} | Resampling: {sampling_name}")
            print(f"  Train size: {len(X_tr)} | "
                  f"Class dist: {pd.Series(y_tr).value_counts().to_dict()}")
            print(f"{'─'*60}")

            best_h, (recall, gmean, auc) = find_best_h(
                clf,
                X_tr, y_tr,
                X_unl_sub,
                X_te_sub, y_test,
                clf_name
            )

            results.append({
                'Fold'         : fold,
                'Normalization': norm_name,
                'Subset'       : subset_name,
                'Classifier'   : clf_name,
                'Resampling'   : sampling_name,
                'Best_h'       : best_h,
                'Recall'       : round(recall, 2),
                'G-Mean'       : round(gmean, 2),
                'AUC'          : round(auc, 2)
            })

            print(f"\n  ✓ {clf_name} | {sampling_name} | {norm_name} | {subset_name}")
            print(f"    Best h={best_h} | Recall={recall:.2f}% | "
                  f"G-Mean={gmean:.2f}% | AUC={auc:.2f}%")

    return pd.DataFrame(results)


# ═════════════════════════════════════════════════════════════
# STEP 9: RUN SINGLE FOLD ONLY
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print(f"STEP 9: RUNNING FOLD {RUN_ONLY_FOLD} OF 5")
print("="*60)

N_FOLDS = 5
skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
X_all   = labeled[all_features]
y_all   = labeled[target]

fold_results = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all), 1):
    if fold != RUN_ONLY_FOLD:
        continue  # skip all other folds

    print(f"\n{'='*60}")
    print(f"  FOLD {fold}/{N_FOLDS}")
    print(f"{'='*60}")

    X_train_raw = X_all.iloc[train_idx].reset_index(drop=True)
    X_test_raw  = X_all.iloc[test_idx].reset_index(drop=True)
    y_train     = y_all.iloc[train_idx].reset_index(drop=True)
    y_test      = y_all.iloc[test_idx].reset_index(drop=True)

    print(f"  Train: {len(X_train_raw)} | Test: {len(X_test_raw)}")
    print(f"  Train dist: {y_train.value_counts().to_dict()}")
    print(f"  Test dist : {y_test.value_counts().to_dict()}")

    # Normalize per fold (fit on train only)
    normZ         = Normalizer()
    X_train_Z     = normZ.fit_transform(X_train_raw, columns_to_normalize, method='zscore')
    X_test_Z      = normZ.transform(X_test_raw,      columns_to_normalize)
    X_unlabeled_Z = normZ.transform(unlabeled[all_features], columns_to_normalize)

    normM         = Normalizer()
    X_train_M     = normM.fit_transform(X_train_raw, columns_to_normalize, method='minmax')
    X_test_M      = normM.transform(X_test_raw,      columns_to_normalize)
    X_unlabeled_M = normM.transform(unlabeled[all_features], columns_to_normalize)

    experiment_configs = {
        'Z-Score': (X_train_Z, X_test_Z, X_unlabeled_Z),
        'Min-Max': (X_train_M, X_test_M, X_unlabeled_M),
    }

    for subset_name, features in feature_subsets.items():
        for norm_name, (X_tr, X_te, X_unl) in experiment_configs.items():
            results = run_experiment(
                X_tr, y_train,
                X_unl, X_te, y_test,
                norm_name, subset_name, features,
                fold=fold
            )
            fold_results.append(results)

            # Checkpoint save after every combination
            pd.concat(fold_results, ignore_index=True).to_csv(
                OUTPUT_FILE, index=False
            )
            print(f"  ✓ Checkpoint saved: Fold {fold} | {subset_name} | {norm_name}")

# ═════════════════════════════════════════════════════════════
# STEP 10: SAVE FOLD RESULTS
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print(f"STEP 10: FOLD {RUN_ONLY_FOLD} RESULTS")
print("="*60)

final = pd.concat(fold_results, ignore_index=True)
final.to_csv(OUTPUT_FILE, index=False)
print(f"\n✓ Results saved to {OUTPUT_FILE}")
print(final.to_string(index=False))

# ═════════════════════════════════════════════════════════════
# MERGE SCRIPT — run this after ALL 5 folds are complete
# ═════════════════════════════════════════════════════════════
# import pandas as pd, glob
#
# files    = sorted(glob.glob(f'{OUT_PATH}/ssstr_fold_*.csv'))
# combined = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
# combined.to_csv(f'{OUT_PATH}/ssstr_cv_all_folds.csv', index=False)
#
# avg_results = combined.groupby(
#     ['Normalization', 'Subset', 'Classifier', 'Resampling']
# )[['Recall', 'G-Mean', 'AUC']].agg(['mean', 'std']).round(2)
#
# avg_results.columns = ['Recall_mean', 'Recall_std',
#                        'GMean_mean',  'GMean_std',
#                        'AUC_mean',    'AUC_std']
# avg_results = avg_results.reset_index()
# avg_results.to_csv(f'{OUT_PATH}/ssstr_cv_results.csv', index=False)
# print(avg_results.to_string(index=False))


CONFIG
  Running fold     : 5 of 5
  Labeled fraction : 10%
  Add noise        : True (level=0.3)
  Labelled path    : /kaggle/input/datasets/mahsaheidary/labelled-dataset/Labelled.csv
  Unlabelled path  : /kaggle/input/datasets/mahsaheidary/unlabelled-dataset/Unlabelled.csv
  Output file      : /kaggle/working/ssstr_fold_5.csv

STEP 1: LOADING DATA
Full labeled dataset : (6873, 11)
Unlabeled dataset    : (23877, 10)
Full class dist:
Fake
1    4020
0    2853
Name: count, dtype: int64

[Option 1] Using 10% as labeled → 687 rows
  Unlabeled pool now: 30063 rows
  Labeled class dist:
Fake
1    402
0    285
Name: count, dtype: int64

[Option 2] Gaussian noise added to: ['likesCount', 'videoCount', 'followerCount', 'followingCount']
  Noise level: 0.3 × feature std

STEP 2: FEATURE SELECTION (RFE-SVM + BIT-FLIP)

── Running RFE-SVM ──

Feature Rankings (1 = most important):
            Feature  Ranking
Nickname_Complexity        1
         videoCount        2
                Bio        3
 

In [4]:
# ═════════════════════════════════════════════════════════════
# MERGE SCRIPT — run this after ALL 5 folds are complete
# ═════════════════════════════════════════════════════════════
import pandas as pd, glob
files    = sorted(glob.glob(f'{OUT_PATH}/ssstr_fold_*.csv'))
combined = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
combined.to_csv(f'{OUT_PATH}/ssstr_cv_all_folds.csv', index=False)

avg_results = combined.groupby(
     ['Normalization', 'Subset', 'Classifier', 'Resampling']
 )[['Recall', 'G-Mean', 'AUC']].agg(['mean', 'std']).round(2)

avg_results.columns = ['Recall_mean', 'Recall_std',
                       'GMean_mean',  'GMean_std',
                       'AUC_mean',    'AUC_std']
avg_results = avg_results.reset_index()
avg_results.to_csv(f'{OUT_PATH}/ssstr_cv_results.csv', index=False)
print(avg_results.to_string(index=False))

Normalization        Subset Classifier Resampling  Recall_mean  Recall_std  GMean_mean  GMean_std  AUC_mean  AUC_std
      Min-Max    3_features         AB      CBUTE        87.50        1.77       89.33       0.31     91.99     1.37
      Min-Max    3_features         AB       NORS        87.50        1.77       88.90       0.29     93.50     2.41
      Min-Max    3_features         AB      SMOTE        87.50        1.77       89.33       0.31     92.40     0.70
      Min-Max    3_features       CART      CBUTE        88.12        0.88       85.65       4.89     85.73     4.78
      Min-Max    3_features       CART       NORS        90.62        0.88       85.42       4.84     85.66     4.52
      Min-Max    3_features       CART      SMOTE        90.00        0.00       86.02       6.49     86.22     6.20
      Min-Max    3_features         GB      CBUTE        87.50        1.77       90.20       0.91     93.26     1.87
      Min-Max    3_features         GB       NORS        83.12  

In [5]:
import os
files = os.listdir('/kaggle/working/')
print(files)

['ssstr_cv_results.csv', 'ssstr_fold_5.csv', '.virtual_documents', 'ssstr_fold_4.csv', 'ssstr_cv_all_folds.csv']


In [2]:
print('/kaggle/working/ssstr_cv_results.csv')

/kaggle/working/ssstr_cv_results.csv


In [3]:
from IPython.display import FileLink
FileLink('ssstr_cv_all_folds.csv')

/kaggle/working/ssstr_cv_all_folds.csv

In [ ]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import recall_score, roc_auc_score, precision_score, f1_score
from sklearn.cluster import KMeans
from imblearn.over_sampling import SMOTE
import os
import copy
import warnings
warnings.filterwarnings('ignore')

# ═════════════════════════════════════════════════════════════
# CONFIG — change these per session
# ═════════════════════════════════════════════════════════════
RUN_ONLY_FOLD    = 1       # ← SET THIS: 1, 2, 3, 4, or 5
LABELED_FRACTION = 0.10
ADD_NOISE        = True
NOISE_LEVEL      = 0.3
NOISE_COLS       = ['likesCount', 'videoCount', 'followerCount', 'followingCount']
H_VALUES         = [0.1, 0.2, 0.3, 0.5]

LABELLED_PATH   = '/kaggle/input/datasets/mahsaheidary/labelled-dataset/Labelled.csv'
UNLABELLED_PATH = '/kaggle/input/datasets/mahsaheidary/unlabelled-dataset/Unlabelled.csv'
OUT_PATH        = '/kaggle/working'

OUTPUT_FILE = f'{OUT_PATH}/ssstr_fold_{RUN_ONLY_FOLD}.csv'

print("\n" + "="*60)
print("CONFIG")
print("="*60)
print(f"  Running fold     : {RUN_ONLY_FOLD} of 5")
print(f"  Labeled fraction : {LABELED_FRACTION*100:.0f}%")
print(f"  Add noise        : {ADD_NOISE} (level={NOISE_LEVEL})")
print(f"  Output file      : {OUTPUT_FILE}")
print("="*60)

# ═════════════════════════════════════════════════════════════
# STEP 1: LOAD DATA
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 1: LOADING DATA")
print("="*60)

labeled_full = pd.read_csv(LABELLED_PATH)
unlabeled    = pd.read_csv(UNLABELLED_PATH)

all_features = [
    'likesCount', 'videoCount', 'followerCount', 'followingCount',
    'verified', 'Bio', 'Has_Contact_Info', 'Jaro_Similarity', 'Nickname_Complexity'
]
target = 'Fake'

print(f"Full labeled dataset : {labeled_full.shape}")
print(f"Unlabeled dataset    : {unlabeled.shape}")
print(f"Full class dist:\n{labeled_full[target].value_counts()}")

# Reduce labeled pool
if LABELED_FRACTION < 1.0:
    labeled, extra_unlabeled = train_test_split(
        labeled_full,
        train_size=LABELED_FRACTION,
        stratify=labeled_full[target],
        random_state=42
    )
    labeled         = labeled.reset_index(drop=True)
    extra_unlabeled = extra_unlabeled[all_features].reset_index(drop=True)
    unlabeled       = pd.concat(
        [unlabeled[all_features], extra_unlabeled], ignore_index=True
    )
    print(f"\n[Option 1] Using {LABELED_FRACTION*100:.0f}% as labeled → {len(labeled)} rows")
    print(f"  Unlabeled pool now: {len(unlabeled)} rows")
    print(f"  Labeled class dist:\n{labeled[target].value_counts()}")
else:
    labeled = labeled_full.copy()

# Add Gaussian noise
if ADD_NOISE:
    np.random.seed(42)
    for col in NOISE_COLS:
        std = labeled[col].std()
        labeled[col]   = labeled[col]   + np.random.normal(0, std * NOISE_LEVEL, size=len(labeled))
        unlabeled[col] = unlabeled[col] + np.random.normal(0, std * NOISE_LEVEL, size=len(unlabeled))
    print(f"\n[Option 2] Gaussian noise added to: {NOISE_COLS}")
else:
    print("\n[Option 2] No noise added")

# ═════════════════════════════════════════════════════════════
# STEP 2: RFE-SVM FEATURE SELECTION + BIT-FLIP
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 2: FEATURE SELECTION (RFE-SVM + BIT-FLIP)")
print("="*60)

X_fs = labeled[all_features]
y_fs = labeled[target]

scaler_fs   = StandardScaler()
X_fs_scaled = pd.DataFrame(scaler_fs.fit_transform(X_fs), columns=all_features)

print("\n── Running RFE-SVM ──")
svm = SVC(kernel='linear', random_state=42)
rfe = RFE(estimator=svm, n_features_to_select=1, step=1)
rfe.fit(X_fs_scaled, y_fs)

feature_ranking = pd.DataFrame({
    'Feature': all_features,
    'Ranking': rfe.ranking_
}).sort_values('Ranking')

print("\nFeature Rankings (1 = most important):")
print(feature_ranking.to_string(index=False))

ranked_features = feature_ranking['Feature'].tolist()
subset_3 = ranked_features[:3]
subset_5 = ranked_features[:5]
subset_7 = ranked_features[:7]

print(f"\n3-feature subset: {subset_3}")
print(f"5-feature subset: {subset_5}")
print(f"7-feature subset: {subset_7}")

feature_subsets = {
    '3_features': subset_3,
    '5_features': subset_5,
    '7_features': subset_7,
}

print("\n" + "="*60)
print("FEATURE SUBSETS SUMMARY")
print("="*60)
for name, feats in feature_subsets.items():
    print(f"  {name:15s}: {feats}")

# ═════════════════════════════════════════════════════════════
# STEP 3: NORMALIZER CLASS
# ═════════════════════════════════════════════════════════════
columns_to_normalize = [
    'likesCount', 'videoCount', 'followerCount', 'followingCount',
    'Jaro_Similarity', 'Nickname_Complexity'
]


class Normalizer:
    def __init__(self):
        self.scaler = None

    def fit_transform(self, df, columns, method='zscore'):
        missing = [c for c in columns if c not in df.columns]
        if missing:
            raise ValueError(f"Columns not found: {missing}")
        self.scaler = StandardScaler() if method == 'zscore' else MinMaxScaler()
        out          = df.copy()
        out[columns] = self.scaler.fit_transform(df[columns])
        return out

    def transform(self, df, columns):
        if self.scaler is None:
            raise ValueError("Scaler not fitted yet.")
        out          = df.copy()
        out[columns] = self.scaler.transform(df[columns])
        return out


# ═════════════════════════════════════════════════════════════
# STEP 4: RESAMPLING FUNCTIONS
# ═════════════════════════════════════════════════════════════

def apply_smote(X_train, y_train, random_state=42):
    y_train = pd.Series(y_train).reset_index(drop=True)
    minority_count = y_train.value_counts().min()
    k = min(5, minority_count - 1)
    if k < 1:
        print("  SMOTE skipped: minority class too small")
        return X_train, y_train
    smote        = SMOTE(random_state=random_state, k_neighbors=k)
    X_res, y_res = smote.fit_resample(X_train, y_train)
    print(f"  After SMOTE: {pd.Series(y_res).value_counts().to_dict()}")
    return X_res, pd.Series(y_res).reset_index(drop=True)


def apply_cbute(X_train, y_train, random_state=42):
    X_train = X_train.reset_index(drop=True)
    y_train = pd.Series(y_train).reset_index(drop=True)

    minority_class = y_train.value_counts().idxmin()
    majority_class = y_train.value_counts().idxmax()

    X_min = X_train[y_train == minority_class]
    X_maj = X_train[y_train == majority_class]
    y_min = y_train[y_train == minority_class]

    n_clusters = max(1, len(X_min))
    kmeans     = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    kmeans.fit(X_maj)

    X_maj_reduced = pd.DataFrame(kmeans.cluster_centers_, columns=X_train.columns)
    y_maj_reduced = pd.Series([majority_class] * n_clusters)

    X_res = pd.concat([X_min.reset_index(drop=True), X_maj_reduced], ignore_index=True)
    y_res = pd.Series(
        pd.concat([y_min.reset_index(drop=True), y_maj_reduced], ignore_index=True)
    ).reset_index(drop=True)

    print(f"  After CBUTE: {y_res.value_counts().to_dict()}")
    return X_res, y_res


# ═════════════════════════════════════════════════════════════
# STEP 5: CLASSIFIERS + METRICS
# ═════════════════════════════════════════════════════════════

classifiers = {
    'CART': DecisionTreeClassifier(random_state=42),
    'RF'  : RandomForestClassifier(n_estimators=20, random_state=42),
    'GB'  : GradientBoostingClassifier(n_estimators=20, random_state=42),
    'AB'  : AdaBoostClassifier(random_state=42, algorithm='SAMME'),
    'KNN' : KNeighborsClassifier(),
    'NB'  : GaussianNB()
}


def g_mean(y_true, y_pred):
    recall = recall_score(y_true, y_pred, zero_division=0)
    tnr    = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    return np.sqrt(recall * tnr)


# ═════════════════════════════════════════════════════════════
# STEP 6: SSSTR ALGORITHM
# ═════════════════════════════════════════════════════════════

def run_ssstr(clf, X_train, y_train, X_unlabeled, X_test, y_test, h, verbose=False):
    L_X = X_train.copy().reset_index(drop=True)
    L_y = pd.Series(y_train).copy().reset_index(drop=True)
    U_X = X_unlabeled.copy().reset_index(drop=True)

    while len(U_X) > 0:
        model = copy.deepcopy(clf)
        model.fit(L_X, L_y)

        if hasattr(model, 'predict_proba'):
            probs        = model.predict_proba(U_X)
            confidence   = probs.max(axis=1)
            pred_col_idx = probs.argmax(axis=1)
            preds        = np.array([model.classes_[i] for i in pred_col_idx])
        else:
            preds      = model.predict(U_X)
            confidence = np.ones(len(preds))

        n_select = max(1, int(h * len(U_X)))
        top_idx  = np.argsort(confidence)[::-1][:n_select]

        L_X = pd.concat([L_X, U_X.iloc[top_idx]],         ignore_index=True)
        L_y = pd.concat([L_y, pd.Series(preds[top_idx])],  ignore_index=True)
        U_X = U_X.drop(index=U_X.index[top_idx]).reset_index(drop=True)

    # Final model
    model = copy.deepcopy(clf)
    model.fit(L_X, L_y)

    y_pred = model.predict(X_test)
    y_prob = (model.predict_proba(X_test)[:, 1]
              if hasattr(model, 'predict_proba') else y_pred)

    # ── Metrics ──────────────────────────────────────────────
    recall    = recall_score(y_test, y_pred, zero_division=0)    * 100
    precision = precision_score(y_test, y_pred, zero_division=0) * 100
    f1        = f1_score(y_test, y_pred, zero_division=0)        * 100
    gmean     = g_mean(y_test, y_pred)                           * 100
    auc       = roc_auc_score(y_test, y_prob)                    * 100

    return recall, precision, f1, gmean, auc


# ═════════════════════════════════════════════════════════════
# STEP 7: FIND BEST h
# ═════════════════════════════════════════════════════════════

def find_best_h(clf, X_train, y_train, X_unlabeled, X_test, y_test, clf_name):
    print(f"\n  Finding best h for {clf_name}...")
    best_h, best_score, best_metrics = None, -1, None

    for h in H_VALUES:
        recall, precision, f1, gmean, auc = run_ssstr(
            clf, X_train, y_train,
            X_unlabeled, X_test, y_test,
            h, verbose=False
        )
        selection_score = (gmean + auc) / 2
        print(f"  h={h} → Recall={recall:.2f}% | Precision={precision:.2f}% | "
              f"F1={f1:.2f}% | G-Mean={gmean:.2f}% | AUC={auc:.2f}% | "
              f"Sel={selection_score:.2f}%")

        if selection_score > best_score:
            best_score   = selection_score
            best_h       = h
            best_metrics = (recall, precision, f1, gmean, auc)

    print(f"\n  ✓ Best h={best_h} (Sel={best_score:.2f}%)")
    return best_h, best_metrics


# ═════════════════════════════════════════════════════════════
# STEP 8: RUN EXPERIMENT PER FEATURE SUBSET
# ═════════════════════════════════════════════════════════════

def run_experiment(X_train, y_train,
                   X_unlabeled, X_test, y_test,
                   norm_name, subset_name, features, fold=None):

    fold_str = f" | Fold {fold}" if fold is not None else ""
    print(f"\n{'='*60}")
    print(f"  Normalization : {norm_name}{fold_str}")
    print(f"  Feature Subset: {subset_name} → {features}")
    print(f"{'='*60}")

    results = []

    X_tr_sub  = X_train[features].reset_index(drop=True)
    X_te_sub  = X_test[features].reset_index(drop=True)
    X_unl_sub = X_unlabeled[features].reset_index(drop=True)
    y_tr_nors = pd.Series(y_train).reset_index(drop=True)

    print(f"\n  Applying resampling on {subset_name} ({len(features)} features)...")
    X_tr_smote, y_tr_smote = apply_smote(X_tr_sub, y_tr_nors)
    X_tr_cbute, y_tr_cbute = apply_cbute(X_tr_sub, y_tr_nors)

    sampling_sets = {
        'NORS' : (X_tr_sub,    y_tr_nors),
        'SMOTE': (X_tr_smote,  y_tr_smote),
        'CBUTE': (X_tr_cbute,  y_tr_cbute)
    }

    for clf_name, clf in classifiers.items():
        for sampling_name, (X_tr, y_tr) in sampling_sets.items():
            print(f"\n{'─'*60}")
            print(f"  Classifier: {clf_name} | Resampling: {sampling_name}")
            print(f"  Train size: {len(X_tr)} | "
                  f"Class dist: {pd.Series(y_tr).value_counts().to_dict()}")
            print(f"{'─'*60}")

            best_h, (recall, precision, f1, gmean, auc) = find_best_h(
                clf, X_tr, y_tr,
                X_unl_sub, X_te_sub, y_test,
                clf_name
            )

            results.append({
                'Fold'         : fold,
                'Normalization': norm_name,
                'Subset'       : subset_name,
                'Classifier'   : clf_name,
                'Resampling'   : sampling_name,
                'Best_h'       : best_h,
                'Recall'       : round(recall,    2),
                'Precision'    : round(precision, 2),
                'F1'           : round(f1,        2),
                'G-Mean'       : round(gmean,     2),
                'AUC'          : round(auc,       2)
            })

            print(f"\n  ✓ {clf_name} | {sampling_name} | {norm_name} | {subset_name}")
            print(f"    Best h={best_h} | Recall={recall:.2f}% | Precision={precision:.2f}% | "
                  f"F1={f1:.2f}% | G-Mean={gmean:.2f}% | AUC={auc:.2f}%")

    return pd.DataFrame(results)


# ═════════════════════════════════════════════════════════════
# STEP 9: RUN SINGLE FOLD ONLY
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print(f"STEP 9: RUNNING FOLD {RUN_ONLY_FOLD} OF 5")
print("="*60)

N_FOLDS = 5
skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
X_all   = labeled[all_features]
y_all   = labeled[target]

fold_results = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all), 1):
    if fold != RUN_ONLY_FOLD:
        continue

    print(f"\n{'='*60}")
    print(f"  FOLD {fold}/{N_FOLDS}")
    print(f"{'='*60}")

    X_train_raw = X_all.iloc[train_idx].reset_index(drop=True)
    X_test_raw  = X_all.iloc[test_idx].reset_index(drop=True)
    y_train     = y_all.iloc[train_idx].reset_index(drop=True)
    y_test      = y_all.iloc[test_idx].reset_index(drop=True)

    print(f"  Train: {len(X_train_raw)} | Test: {len(X_test_raw)}")
    print(f"  Train dist: {y_train.value_counts().to_dict()}")
    print(f"  Test dist : {y_test.value_counts().to_dict()}")

    normZ         = Normalizer()
    X_train_Z     = normZ.fit_transform(X_train_raw, columns_to_normalize, method='zscore')
    X_test_Z      = normZ.transform(X_test_raw,      columns_to_normalize)
    X_unlabeled_Z = normZ.transform(unlabeled[all_features], columns_to_normalize)

    normM         = Normalizer()
    X_train_M     = normM.fit_transform(X_train_raw, columns_to_normalize, method='minmax')
    X_test_M      = normM.transform(X_test_raw,      columns_to_normalize)
    X_unlabeled_M = normM.transform(unlabeled[all_features], columns_to_normalize)

    experiment_configs = {
        'Z-Score': (X_train_Z, X_test_Z, X_unlabeled_Z),
        'Min-Max': (X_train_M, X_test_M, X_unlabeled_M),
    }

    for subset_name, features in feature_subsets.items():
        for norm_name, (X_tr, X_te, X_unl) in experiment_configs.items():
            results = run_experiment(
                X_tr, y_train,
                X_unl, X_te, y_test,
                norm_name, subset_name, features,
                fold=fold
            )
            fold_results.append(results)

            pd.concat(fold_results, ignore_index=True).to_csv(
                OUTPUT_FILE, index=False
            )
            print(f"  ✓ Checkpoint saved: Fold {fold} | {subset_name} | {norm_name}")

# ═════════════════════════════════════════════════════════════
# STEP 10: SAVE FOLD RESULTS
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print(f"STEP 10: FOLD {RUN_ONLY_FOLD} RESULTS")
print("="*60)

final = pd.concat(fold_results, ignore_index=True)
final.to_csv(OUTPUT_FILE, index=False)
print(f"\n✓ Results saved to {OUTPUT_FILE}")
print(final.to_string(index=False))

# ═════════════════════════════════════════════════════════════
# MERGE SCRIPT — run after ALL 5 folds complete
# ═════════════════════════════════════════════════════════════
# import pandas as pd, glob
#
# files    = sorted(glob.glob(f'{OUT_PATH}/ssstr_fold_*.csv'))
# combined = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
# combined.to_csv(f'{OUT_PATH}/ssstr_cv_all_folds.csv', index=False)
#
# avg_results = combined.groupby(
#     ['Normalization', 'Subset', 'Classifier', 'Resampling']
# )[['Recall', 'Precision', 'F1', 'G-Mean', 'AUC']].agg(['mean', 'std']).round(2)
#
# avg_results.columns = [
#     'Recall_mean',    'Recall_std',
#     'Precision_mean', 'Precision_std',
#     'F1_mean',        'F1_std',
#     'GMean_mean',     'GMean_std',
#     'AUC_mean',       'AUC_std'
# ]
# avg_results = avg_results.reset_index()
# avg_results.to_csv(f'{OUT_PATH}/ssstr_cv_results.csv', index=False)
# print(avg_results.to_string(index=False))